# 재현 패키지 메인 노트북 — 제조 전력 24시간 예측과 피크 경보

**제6회 K-인공지능 제조데이터 분석 경진대회 ⑤ 제조 생산데이터 기반 전력사용량 예측 및 최대피크 위험조건 분석**

이 노트북 **하나를 위에서 아래로 실행**하면 데이터 전처리 → 모델 학습 → 8~9월 예측 → 결과 파일 생성 → 자체 검증까지 자동으로 끝난다(CPU, 약 3분).

## 무엇을 예측하나
- 매 정각을 기준으로 **향후 24시간의 15분 전력 96개**를 한꺼번에 예측한다. 생산계획(앞으로 24시간의 생산량)은 예측 시점에 알려져 있다고 가정한다.
- **피크** = 15분 전력 **176 이상**(1~6월 원자료 15분 값의 상위 5% 경계). 피크 경보 = 예측값이 경보 기준 이상일 때.

## 모델 3종 (이 노트북이 모두 직접 학습한다)
| 이름 | 설명 |
|---|---|
| **튜닝 잔차 GRU** | 1주 전 같은 시간 값을 기준으로 두고 그 위의 오차(잔차)를 예측하는 양방향 2층 GRU. 피크 칸에 손실 가중치 3. 9 epoch, seed 42 |
| **현재 Q1** | LightGBM 분위수 회귀(q=0.90), 공식 입력 30개. 피크를 거의 놓치지 않지만 잘못된 경보가 많다 |
| **새 Q1** | 같은 LightGBM에 ① 3단 규칙 값·단계(시간순 OOF) ② 상태 전환 피처(휴무·복귀 여부 등 15개)를 더하고, **q=0.75**로 낮추고, **월요일마다 재학습**하며, **경보 기준을 매주 최근 4주 결과로 갱신**한다 |

## 최종 출력 3종 (목적별로 분리)
| 출력 | 만드는 방법 | 쓰임새 |
|---|---|---|
| **곡선**(15분 전력 수치 예측) | 3단 규칙 값 + LightGBM 잔차(L1), 월요일마다 재학습. 목표 날에 생산계획이 없는 휴무일은 과거 휴무일의 시각별 중앙값 프로파일 | 부하 계획, 수치 정확도(MAE) |
| **경보 / 주의**(피크 여부) | 새 Q1 예측값 ≥ 주별 경보 기준(재현율 목표 .85) → 경보, ≥ 더 낮은 기준(재현율 목표 .95) → 주의 | 사전 경보, 단계별 조치 |
| **피크 확률** | 새 Q1 예측값을 최근 4주 결과로 로지스틱 보정 → P(전력 ≥ 176) | 위험도 표시, 확률 기반 의사결정 |

## 평가 구간과 정직한 표기
| 구간 | 학습에 쓰는 데이터 | 의미 |
|---|---|---|
| 7월(검증) | 타깃이 7/1 이전에 끝난 창(3,985개) | 모델·설정을 고른 구간 |
| 8~9월(테스트) | 타깃이 8/1 이전에 끝난 창(4,635개) | **개발 중 이미 본 구간 → "개발 검증"으로만 읽는다.** 완전히 새로운 미사용 기간은 없다 |

- 새 Q1의 q=0.75·경보 기준 갱신 방식은 L5·L6 실험을 거쳐 정했다. 7월 F2만으로 고르면 q=0.90이 뽑혔고 8~9월에서 나빴다 → 이후 "7월 재현율 ≥ .85 중 정밀도 최대" 기준(q=0.75)으로 바꿨다. **이 기준 변경은 결과를 본 뒤의 사후 변경**이며, 자세한 실험은 `notebooks/` 의 실험 노트북(09~12)에 있다.
- 사용한 정보: **기준시각 이전의 실적**과 **기준시각부터 23시간의 생산계획**뿐이다. 마지막 장에서 미래 값을 바꿔도 피처가 변하지 않는지 직접 검사한다.

## 결과 파일 (`outputs/`)
- `test_predictions_8_9월.csv`: 8~9월 15분 칸별 실제값, 각 모델 예측, 경보 기준, 경보 여부(**테스트데이터 예측결과 파일**)
- `validation_predictions_7월.csv`, `metrics_summary.csv`(정밀도·재현율·F1·F2·MAE·정확도), `metrics_curve.csv`(곡선 MAE·RMSE·평균 오차), `metrics_alarm_levels.csv`(경보·주의 단계), `metrics_probability.csv`(Brier·ECE 등), `verification_checks.csv`, `run_log.txt`, `figures/`

## 실행 방법
프로젝트 루트(이 노트북이 있는 폴더)에서 `jupyter nbconvert --to notebook --execute --inplace main_pipeline.ipynb`, 또는 주피터에서 위에서 아래로 모두 실행. 환경은 `requirements.txt`.

## 0. 환경과 시드

In [ ]:
import os, sys, platform, random, json, time, types, io, contextlib, warnings, shutil, tempfile
os.environ["PYTHONHASHSEED"] = "42"
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import lightgbm as lgb
import torch
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
_fonts = {f.name for f in fm.fontManager.ttflist}
_kf = next((f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"] if f in _fonts), None)
if _kf: plt.rcParams["font.family"] = _kf
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

ROOT = Path.cwd()
assert (ROOT / "data/okm_augumented_2021.csv").exists(), "프로젝트 루트(data/ 폴더가 있는 곳)에서 실행하세요"
OUTDIR = ROOT / "outputs"; (OUTDIR / "figures").mkdir(parents=True, exist_ok=True)
(OUTDIR / "run_log.txt").write_text("", encoding="utf-8")


def log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(OUTDIR / "run_log.txt", "a", encoding="utf-8") as f: f.write(line + "\n")


T_START = time.time()
log(f"Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | lightgbm {lgb.__version__} | torch {torch.__version__} | seed {SEED}")

## 1. 데이터 전처리
원본 `data/okm_augumented_2021.csv`(2021-01-01~09-14, 시간별 6,168행, 행마다 15분 전력 4개)에서 **시간 오류 48행을 복구**하고, **기준시각별 24시간 입력(생산계획, 달력, 1·2주 전 전력·생산량)과 정답(15분 전력 96개)**을 만든다. 학습·검증·테스트를 시간순으로 나누고 정답 24시간이 경계를 넘는 창은 따로 표시한다. 아래 셀은 팀 공통 전처리(`01_preprocessing.ipynb`)의 코드를 그대로 옮긴 것이다.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print  # Jupyter 없이 스크립트로 검증할 때 사용

ROOT = Path.cwd()
if not (ROOT / 'data/okm_augumented_2021.csv').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data/okm_augumented_2021.csv'
OUT = ROOT / 'results/preprocessing_24h'
POWER_COLS = ['15분', '30분', '45분', '60분']
MINUTES = [15, 30, 45, 60]
HORIZON = 24
VALID_START = pd.Timestamp('2021-07-01')
TEST_START = pd.Timestamp('2021-08-01')
SUSPECT_DATES = pd.to_datetime(['2021-07-13', '2021-07-15'])
# 기존 전처리에서 사용한 대한민국 2021년 공휴일 달력을 유지합니다.
HOLIDAYS = pd.to_datetime([
    '2021-01-01', '2021-02-11', '2021-02-12', '2021-02-13',
    '2021-03-01', '2021-05-05', '2021-05-19', '2021-06-06',
    '2021-08-15', '2021-08-16', '2021-09-20', '2021-09-21',
    '2021-09-22', '2021-10-03', '2021-10-04', '2021-10-09',
    '2021-10-11', '2021-12-25'
])
raw = pd.read_csv(DATA_PATH)
print('원본:', raw.shape)

## 1. 시간 복구와 원본 검증
날짜마다 24개 행이 있고 정상 시간과 행 순서가 일치하는지 확인한 뒤, 잘못된 시간만 0~23으로 복구합니다. 날짜를 삭제한 상태에서 shift하지 않습니다. 모델에 쓰는 생산계획·전력에 결측이나 무한값이 있으면 중단하여 확인합니다.

In [ ]:
data = raw.copy()
data['date'] = pd.to_datetime(data['날짜'].astype(str), format='%Y%m%d')
data['original_hour'] = data['시간']
row_hour = data.groupby('date', sort=False).cumcount()
valid_hour = data['시간'].between(0, 23) & data['시간'].eq(data['시간'].round())
assert data.groupby('date').size().eq(24).all(), '날짜별 행 수 확인 필요'
assert data.loc[valid_hour, '시간'].eq(row_hour[valid_hour]).all(), '원본 행 순서 확인 필요'
data['time_repaired'] = ~valid_hour
data.loc[~valid_hour, '시간'] = row_hour[~valid_hour]
data['origin'] = data['date'] + pd.to_timedelta(data['시간'], unit='h')
data = data.sort_values('origin').reset_index(drop=True)
assert data['origin'].is_unique
assert data['origin'].diff().dropna().eq(pd.Timedelta(hours=1)).all()
assert np.isfinite(data[POWER_COLS + ['생산량']].to_numpy(dtype=float)).all()
assert data[POWER_COLS + ['생산량']].ge(0).all().all()
data['suspect_date'] = data['date'].isin(SUSPECT_DATES)
# 11시 행 전체를 포함: 행 시작시각 기준 [8/28 18시, 8/29 12시).
data['special_period'] = data['origin'].ge('2021-08-28 18:00') & data['origin'].lt('2021-08-29 12:00')
calendar = pd.DataFrame({
    'month': data['origin'].dt.month,
    'day': data['origin'].dt.day,
    'hour': data['origin'].dt.hour,
    'weekday': data['origin'].dt.dayofweek,
    'weekend': data['origin'].dt.dayofweek.ge(5).astype(int),
    'holiday': data['date'].isin(HOLIDAYS).astype(int),
})
display(data.loc[data['time_repaired'], ['date', 'original_hour', '시간']].head())
print('복구 행:', int(data['time_repaired'].sum()))
print('범위:', data['origin'].min(), '~', data['origin'].max())

## 2. 기준시각별 24시간 입력과 정답
`h00`은 기준시각의 시간 행, `h23`은 23시간 뒤 행입니다. `w1`, `w2`는 각각 해당 대상 행의 168·336시간 전입니다. 최근 전력 4개는 모든 미래 시간에 대해 동일한 기준시각에서 확보한 값입니다.

머신러닝용 `X`는 펼친 표이고, 딥러닝용 `sequence`는 같은 미래·주간 특성을 `(표본, 24, 특성)`으로 묶습니다. 최근 전력은 별도의 `(표본, 4)` 배열입니다. 정답의 펼침 순서는 시간 → 15·30·45·60분입니다.

In [ ]:
# 시간축의 연속성을 검증했으므로 정수 위치로 정확한 시간 블록을 조회합니다.
starts = np.arange(336, len(data) - HORIZON + 1)
target_pos = starts[:, None] + np.arange(HORIZON)
w1_pos, w2_pos = target_pos - 168, target_pos - 336
power = data[POWER_COLS].to_numpy(dtype=float)
production = data['생산량'].to_numpy(dtype=float)
holiday = calendar['holiday'].to_numpy()
plan = production[target_pos]
sequence_parts = [plan[..., None]]
sequence_names = ['planned_production']
for name in calendar.columns:
    sequence_parts.append(calendar[name].to_numpy()[target_pos][..., None])
    sequence_names.append(name)
for week, pos in [(1, w1_pos), (2, w2_pos)]:
    sequence_parts.extend([
        power[pos], production[pos][..., None], holiday[pos][..., None],
        (plan - production[pos])[..., None],
        ((plan > 0) != (production[pos] > 0))[..., None],
        (holiday[target_pos] != holiday[pos])[..., None],
    ])
    sequence_names.extend([
        *[f'w{week}_power_{m}' for m in MINUTES],
        f'w{week}_production', f'w{week}_holiday',
        f'w{week}_production_difference', f'w{week}_production_state_mismatch',
        f'w{week}_holiday_mismatch',
    ])
sequence = np.concatenate(sequence_parts, axis=2).astype(float)
recent = power[starts - 1]
targets = power[target_pos]
recent_names = [f'recent_previous_hour_{m}' for m in MINUTES]
feature_names = recent_names + [f'h{h:02d}_{name}' for h in range(HORIZON) for name in sequence_names]
target_names = [f'h{h:02d}_power_{m}' for h in range(HORIZON) for m in MINUTES]
sample_ids = pd.Index(starts, name='sample_id')
X = pd.DataFrame(np.concatenate([recent, sequence.reshape(len(starts), -1)], axis=1), index=sample_ids, columns=feature_names)
y = pd.DataFrame(targets.reshape(len(starts), -1), index=sample_ids, columns=target_names)
print('X:', X.shape, '| y:', y.shape, '| sequence:', sequence.shape)

## 3. 시간순 분할과 공통 평가 표시
학습·검증·테스트 각각에 정답 24시간이 완전히 들어오는 표본만 사용합니다. 경계를 가로지르는 표본은 별도로 표시합니다. 검증·테스트의 과거 입력은 이전 분할을 참조할 수 있습니다.

- 주 검증: 정답 구간이 7/13·7/15와 겹치지 않는 표본
- 보조 검증: 전체 표본, 그리고 정답·최근 전력·1·2주 전 입력 모두 의심 날짜를 참조하지 않는 표본
- 운영 시각은 미정이므로 모든 정각을 저장하며, 추후 `origin_hour`로 공통 선택합니다.
- 최종 1~7월 재학습 데이터는 **후보**로 저장합니다. 이상 날짜 포함 정책은 아직 확정하지 않았으므로 `primary_eligible` 또는 `history_clean_eligible` 등을 선택한 후 학습합니다.

특정 시간의 성능을 평가할 때는 아래 `target_masks`를 정답과 같은 순서로 사용합니다. 표본 전체 포함 여부와 시간별 포함 여부를 구분합니다.

In [ ]:
meta = pd.DataFrame(index=sample_ids)
meta['origin'] = data['origin'].iloc[starts].to_numpy()
meta['target_end'] = meta['origin'] + pd.Timedelta(hours=HORIZON)
meta['origin_hour'] = meta['origin'].dt.hour
suspect = data['suspect_date'].to_numpy()
special = data['special_period'].to_numpy()
meta['target_suspect'] = suspect[target_pos].any(axis=1)
meta['recent_suspect'] = suspect[starts - 1]
meta['w1_suspect'] = suspect[w1_pos].any(axis=1)
meta['w2_suspect'] = suspect[w2_pos].any(axis=1)
meta['primary_eligible'] = ~meta['target_suspect']
meta['history_clean_eligible'] = ~meta[['target_suspect', 'recent_suspect', 'w1_suspect', 'w2_suspect']].any(axis=1)
meta['special_period_overlap'] = special[target_pos].any(axis=1)
meta['split'] = 'boundary_excluded'
meta.loc[meta['target_end'].le(VALID_START), 'split'] = 'train'
meta.loc[meta['origin'].ge(VALID_START) & meta['target_end'].le(TEST_START), 'split'] = 'validation'
meta.loc[meta['origin'].ge(TEST_START), 'split'] = 'test'
meta['final_retrain_candidate'] = meta['target_end'].le(TEST_START)
split_masks = {name: meta['split'].eq(name).to_numpy() for name in ['train', 'validation', 'test']}
split_masks['final_retrain_candidates'] = meta['final_retrain_candidate'].to_numpy()

# 기준은 겹친 윈도우가 아니라 1~6월 원본의 각 관측값을 딱 한 번 사용합니다.
threshold_source = data.loc[data['origin'].lt(VALID_START), POWER_COLS].to_numpy().ravel()
peak_threshold = float(np.quantile(threshold_source, 0.95))
hour_masks = {
    'suspect_target': suspect[target_pos],
    'special_period': special[target_pos],
    'weekend_production': (calendar['weekend'].to_numpy()[target_pos] == 1) & (plan > 0),
    'holiday_production': (holiday[target_pos] == 1) & (plan > 0),
    'zero_production': plan == 0,
    'production_both_weeks_zero': (plan > 0) & (production[w1_pos] == 0) & (production[w2_pos] == 0),
}
target_masks = {name: np.repeat(values, 4, axis=1) for name, values in hour_masks.items()}
for name, lo, hi in [('horizon_0_3', 0, 3), ('horizon_3_12', 3, 12), ('horizon_12_24', 12, 24)]:
    hours = (np.arange(24) >= lo) & (np.arange(24) < hi)
    target_masks[name] = np.broadcast_to(np.repeat(hours, 4), y.shape).copy()
target_masks['actual_peak'] = y.to_numpy() >= peak_threshold
split_summary = meta.groupby('split').agg(samples=('origin', 'size'), first_origin=('origin', 'min'), last_origin=('origin', 'max'), primary_samples=('primary_eligible', 'sum'), history_clean_samples=('history_clean_eligible', 'sum'))
display(split_summary)
print('고정 피크 기준(q95):', peak_threshold)

## 4. 정렬·누출·분할 검증
과거 블록의 마지막 관측도 기준시각 이전인지 확인합니다. 첫 표본과 마지막 표본을 포함해 원본 시각으로 직접 조회하여, 배열 구성과 정답 순서를 검증합니다.

In [ ]:
assert X.index.equals(y.index) and y.index.equals(meta.index)
assert np.isfinite(X.to_numpy()).all() and np.isfinite(y.to_numpy()).all()
assert X.columns.is_unique and y.columns.is_unique
assert meta['origin'].iloc[0] == pd.Timestamp('2021-01-15')
assert meta.loc[meta['split'].eq('train'), 'origin'].max() == pd.Timestamp('2021-06-30')
for pos in [w1_pos, w2_pos]:
    latest_observed = data['origin'].to_numpy()[pos[:, -1]] + np.timedelta64(60, 'm')
    assert (latest_observed <= meta['origin'].to_numpy()).all()
assert (data['origin'].to_numpy()[starts - 1] + np.timedelta64(60, 'm') == meta['origin'].to_numpy()).all()
lookup = data.set_index('origin')
for i in [0, len(starts) // 2, len(starts) - 1]:
    origin = meta['origin'].iloc[i]
    times = pd.date_range(origin, periods=24, freq='h')
    np.testing.assert_array_equal(targets[i], lookup.loc[times, POWER_COLS].to_numpy())
    np.testing.assert_array_equal(recent[i], lookup.loc[origin - pd.Timedelta(hours=1), POWER_COLS].to_numpy(dtype=float))
    for week in [1, 2]:
        cols = [sequence_names.index(f'w{week}_power_{m}') for m in MINUTES]
        np.testing.assert_array_equal(sequence[i][:, cols], lookup.loc[times - pd.Timedelta(weeks=week), POWER_COLS].to_numpy())
for name, mask in split_masks.items():
    assert mask.any(), f'{name}: 빈 분할'
assert meta.loc[split_masks['train'], 'target_end'].le(VALID_START).all()
assert meta.loc[split_masks['validation'], 'origin'].ge(VALID_START).all()
assert meta.loc[split_masks['validation'], 'target_end'].le(TEST_START).all()
assert meta.loc[split_masks['test'], 'origin'].ge(TEST_START).all()
assert meta['target_end'].max() == data['origin'].max() + pd.Timedelta(hours=1)
assert len(threshold_source) == 181 * 24 * 4
np.testing.assert_array_equal(X.iloc[:, 4:].to_numpy().reshape(sequence.shape), sequence)
print('원본 대응·과거 정보 시각·정답 경계·배열 순서 검증 통과')

## 5. 팀 공통 파일 저장
`X_*.csv`, `y_*.csv`, `metadata_*.csv`는 같은 `sample_id` 순서입니다. `arrays_*.npz`에는 같은 입력·정답의 배열, 1·2주 기준선, 시간별 평가 마스크를 저장합니다. `config.json`에 열 순서·피크 기준·원본 해시를 기록합니다.

`hourly_review.csv`는 비용·공장인원 등을 포함한 검토용 자료이며 **모델 입력에 합치지 않습니다**. 최종 재학습 후보에는 6월/7월 경계를 넘는 표본도 포함됩니다. 처음 모델을 선정할 때 이 파일을 사용하면 안 됩니다.

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
for name, mask in split_masks.items():
    for prefix, frame in [('X', X), ('y', y), ('metadata', meta)]:
        frame.loc[mask].to_csv(OUT / f'{prefix}_{name}.csv', encoding='utf-8-sig', index=True)
    np.savez_compressed(
        OUT / f'arrays_{name}.npz',
        sample_id=sample_ids.to_numpy()[mask], sequence=sequence[mask], recent=recent[mask],
        y=targets[mask], baseline_w1=power[w1_pos][mask], baseline_w2=power[w2_pos][mask],
        baseline_weekly_mean=((power[w1_pos] + power[w2_pos]) / 2)[mask],
        **{key: value[mask] for key, value in target_masks.items()},
    )
meta.to_csv(OUT / 'all_sample_metadata.csv', encoding='utf-8-sig')
split_summary.to_csv(OUT / 'split_summary.csv', encoding='utf-8-sig')
data.to_csv(OUT / 'hourly_review.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'feature': feature_names}).to_csv(OUT / 'feature_catalog.csv', index=False, encoding='utf-8-sig')
pd.DataFrame([
    {'target': f'h{h:02d}_power_{m}', 'hour_offset': h, 'minute_in_hour': m, 'observation_offset_minutes': h * 60 + m}
    for h in range(24) for m in MINUTES
]).to_csv(OUT / 'target_catalog.csv', index=False, encoding='utf-8-sig')
config = {
    'source_file': str(DATA_PATH.relative_to(ROOT)),
    'source_sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    'horizon_hours': 24, 'stride_hours': 1, 'history_hours': [168, 336],
    'feature_columns': feature_names, 'sequence_features': sequence_names,
    'recent_features': recent_names, 'target_columns': target_names,
    'flatten_order': 'hour first, then feature / power minute (15,30,45,60)',
    'peak_threshold': peak_threshold, 'peak_comparison': '>=',
    'threshold_source': 'unique raw observations Jan 1 through June 30; fixed after retraining',
    'threshold_observations': len(threshold_source),
    'validation_start': str(VALID_START), 'test_start': str(TEST_START),
    'holiday_dates': [str(d.date()) for d in HOLIDAYS],
    'suspect_dates': [str(d.date()) for d in SUSPECT_DATES],
    'primary_evaluation_mask': 'primary_eligible',
    'strict_evaluation_mask': 'history_clean_eligible',
    'special_period_row_interval': '[2021-08-28 18:00, 2021-08-29 12:00)',
    'operating_hour': None, 'final_retrain_suspect_policy': 'not yet selected; candidates only',
    'availability': '24h production plan known at origin; previous hour 60-minute power immediately available',
    'scaling': None, 'missing_value_policy': 'stop if required production or power is missing',
    'weather': 'excluded', 'random_seed_for_downstream_models': 42,
}
(OUT / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
# 저장 후 다시 읽어 팀원이 불러올 때도 행 순서와 값이 유지되는지 확인합니다.
for name, mask in split_masks.items():
    saved_x = pd.read_csv(OUT / f'X_{name}.csv', index_col='sample_id')
    saved_y = pd.read_csv(OUT / f'y_{name}.csv', index_col='sample_id')
    assert saved_x.index.equals(saved_y.index)
    np.testing.assert_allclose(saved_x.to_numpy(), X.loc[mask].to_numpy())
    with np.load(OUT / f'arrays_{name}.npz') as saved:
        np.testing.assert_array_equal(saved['sample_id'], saved_x.index.to_numpy())
        np.testing.assert_allclose(saved['y'].reshape(len(saved_y), -1), saved_y.to_numpy())
print('저장 및 재로딩 검증 완료:', OUT)

In [ ]:
_ss = pd.read_csv(ROOT / "results/preprocessing_24h/split_summary.csv"); display(_ss)
log("전처리 완료: results/preprocessing_24h/ (arrays_*.npz, metadata_*.csv, hourly_review.csv, config.json)")

## 2. 공통 함수: 15분 칸 피처, 3단 규칙(시간순 OOF), 지표
- **공식 Q1 입력 30개**: 직전 1시간 전력 4개, 달력 6개, 1·2주 전 같은 칸 값 등 11개, 생산계획·차이·불일치 5개, 두 주 평균·최근 통계 4개.
- **3단 규칙**: 이 시간의 가동 여부(생산계획 > 0)가 ① 1주 전과 같으면 1주 전 값 ② 아니면 2주 전과 같으면 2주 전 값 ③ 둘 다 다르면 (가동 여부, 시, 분)별 평균 프로파일. 평균 프로파일은 **각 창 기준일 00시 이전에 확정된 관측만**으로 계산한다(`OOFProfile`, 학습 정답이 섞이지 않는다).
- 지표: 피크는 실제·예측 모두 15분 칸 단위. **F1·F2·정밀도·재현율**과 함께 MAE·평균 오차를 보고, 경보를 한 번도 안 울릴 때의 정확도(`no_alarm_accuracy`)도 함께 낸다(피크가 드물어 정확도만 보면 속는다).

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
PREP = ROOT / "results/preprocessing_24h"
PEAK = 176.0
_SEQ_DEFAULT = ["planned_production", "month", "day", "hour", "weekday", "weekend", "holiday", "w1_power_15", "w1_power_30", "w1_power_45", "w1_power_60",
                "w1_production", "w1_holiday", "w1_production_difference", "w1_production_state_mismatch", "w1_holiday_mismatch",
                "w2_power_15", "w2_power_30", "w2_power_45", "w2_power_60", "w2_production", "w2_holiday", "w2_production_difference",
                "w2_production_state_mismatch", "w2_holiday_mismatch"]
# 전처리 결과가 있으면 config.json을 읽고, 없으면(전처리 전) 기본값으로 모듈을 불러올 수 있게 한다.
CFG = json.loads((PREP / "config.json").read_text(encoding="utf-8")) if (PREP / "config.json").exists() else {}
SEQ = CFG.get("sequence_features", _SEQ_DEFAULT)
assert float(CFG.get("peak_threshold", PEAK)) == PEAK

# 현재 공식 Q1 하이퍼파라미터(03 §10, 06과 동일)
Q1_PARAMS = dict(
    objective="quantile", alpha=0.90, learning_rate=0.03, num_leaves=31, max_depth=-1,
    min_child_samples=50, subsample=0.85, subsample_freq=1, colsample_bytree=1.0,
    reg_alpha=1.0, reg_lambda=10.0, n_estimators=272, random_state=42, n_jobs=-1, verbose=-1,
)

REC = [f"rec_{m}" for m in (15, 30, 45, 60)]
CAL = ["month", "day", "hour", "weekday", "weekend", "holiday"]
HIST = ["w1_same", "w2_same", "w1_production", "w2_production", "w1_holiday", "w2_holiday",
        "w1_holiday_mismatch", "w2_holiday_mismatch", "minute", "lead", "hour_offset"]
PLAN = ["planned_production", "w1_production_difference", "w2_production_difference",
        "w1_production_state_mismatch", "w2_production_state_mismatch"]
EXTRA = ["two_week_mean_same", "rec_mean1h", "rec_std1h", "rec_change1h"]
Q1_FEATURES = REC + CAL + HIST + PLAN + EXTRA          # 현재 공식 Q1 30개


def as_bool(s):
    if s.dtype == bool:
        return s.to_numpy()
    return s.astype(str).str.lower().map({"true": True, "false": False}).to_numpy(bool)


def long_features(arrays, rows=None):
    """공식 Q1과 같은 15분 칸 단위 피처 표(창 × 96칸). 03 §10 / 06 셀 7과 동일 로직."""
    seq = arrays["sequence"] if rows is None else arrays["sequence"][rows]
    recent = arrays["recent"] if rows is None else arrays["recent"][rows]
    seq = seq.astype(np.float32); recent = recent.astype(np.float32); n = len(seq)
    idx = {name: i for i, name in enumerate(SEQ)}
    rep = lambda a: np.repeat(a, 4, axis=1).reshape(-1)
    v = {}
    for j, m in enumerate((15, 30, 45, 60)):
        v[f"rec_{m}"] = np.broadcast_to(recent[:, j, None], (n, 96)).reshape(-1)
    for name in [x for x in SEQ if "_power_" not in x]:
        v[name] = rep(seq[:, :, idx[name]])
    w1 = seq[:, :, [idx[f"w1_power_{m}"] for m in (15, 30, 45, 60)]].reshape(n, 96)
    w2 = seq[:, :, [idx[f"w2_power_{m}"] for m in (15, 30, 45, 60)]].reshape(n, 96)
    v["w1_same"] = w1.reshape(-1); v["w2_same"] = w2.reshape(-1)
    v["two_week_mean_same"] = ((w1 + w2) / 2).reshape(-1)
    v["minute"] = np.tile(np.array([15, 30, 45, 60], np.float32), n * 24)
    v["lead"] = np.tile(np.arange(96, dtype=np.float32), n)
    v["hour_offset"] = np.tile(np.repeat(np.arange(24, dtype=np.float32), 4), n)
    for name, x in {"rec_mean1h": recent.mean(axis=1), "rec_std1h": recent.std(axis=1),
                    "rec_change1h": recent[:, -1] - recent[:, 0]}.items():
        v[name] = np.broadcast_to(x[:, None], (n, 96)).reshape(-1)
    return pd.DataFrame(v, dtype=np.float32)


# ---------------------------------------------------------------- 3단 규칙
def rule_inputs(F):
    return (F["planned_production"].to_numpy() > 0, F["w1_production"].to_numpy() > 0, F["w2_production"].to_numpy() > 0)


def observation_table(sets):
    """주어진 (arrays, meta) 목록에서 '시각별 정답 관측' 표를 만든다(중복 시각 제거, 의심 칸 제외).
    열: ts(15분 구간 끝 시각), day(관측이 확정된 날), plan_on, hour, minute, y."""
    rows = []
    for a, m in sets:
        n = len(m); seq = a["sequence"].astype(np.float32)
        y = a["y"].reshape(n, 96)
        sus = a["suspect_target"].reshape(n, 96).astype(bool)
        h = np.tile(np.repeat(np.arange(24), 4), n); q = np.tile(np.arange(4), n * 24)
        win = np.repeat(np.arange(n), 96)
        origin = pd.to_datetime(m["origin"].to_numpy())[win]
        ts = origin + pd.to_timedelta(h, "h") + pd.to_timedelta(15 * (q + 1), "m")
        ih = SEQ.index("hour"); ip = SEQ.index("planned_production")
        df = pd.DataFrame(dict(ts=ts, plan_on=(seq[win, h, ip] > 0).astype(np.int8),
                               hour=seq[win, h, ih].astype(np.int8), minute=((q + 1) * 15).astype(np.int8),
                               y=y.reshape(-1), sus=sus.reshape(-1)))
        rows.append(df[~df.sus].drop(columns="sus"))
    obs = pd.concat(rows, ignore_index=True).drop_duplicates("ts").sort_values("ts").reset_index(drop=True)
    obs["day"] = (obs.ts - pd.Timedelta(minutes=1)).dt.floor("D")
    return obs


class OOFProfile:
    """기준시각 이전에 이미 확정된 관측만으로 (가동 여부, 시, 분) 평균을 만든다.
    창의 기준일 D에 대해 '관측 day < D'(= D 00:00 이전 관측)만 사용 → 학습 정답이 자기 자신에게 섞이지 않는다."""
    def __init__(self, obs):
        self.days = np.sort(obs.day.unique())
        key = (obs.plan_on.astype(int) * 10000 + obs.hour.astype(int) * 100 + (obs.minute.astype(int) // 15)).to_numpy()
        self.keys = np.unique(key); kidx = {k: i for i, k in enumerate(self.keys)}
        di = {d: i for i, d in enumerate(self.days)}
        S = np.zeros((len(self.days), len(self.keys))); C = np.zeros_like(S)
        for k, d, y in zip(key, obs.day.to_numpy(), obs.y.to_numpy()):
            S[di[d], kidx[k]] += y; C[di[d], kidx[k]] += 1
        self.cS = np.cumsum(S, 0); self.cC = np.cumsum(C, 0); self.kidx = kidx
        self.gS = np.cumsum([obs[obs.day == d].groupby("plan_on").y.sum().reindex([0, 1]).fillna(0).to_numpy() for d in self.days], axis=0)
        self.gC = np.cumsum([obs[obs.day == d].groupby("plan_on").y.count().reindex([0, 1]).fillna(0).to_numpy() for d in self.days], axis=0)

    def lookup(self, origin_day, plan_on, hour, minute):
        """origin_day(셀별 창 기준일), 반환: 프로파일 값(없으면 plan_on 전체 평균, 그것도 없으면 NaN)."""
        pos = np.searchsorted(self.days, np.asarray(origin_day, dtype="datetime64[ns]"), side="left") - 1   # day < D 의 마지막 인덱스
        ok = pos >= 0; p = np.where(ok, pos, 0)
        key = plan_on.astype(int) * 10000 + hour.astype(int) * 100 + (minute.astype(int) // 15)
        ki = np.array([self.kidx.get(k, -1) for k in key]); has = ok & (ki >= 0)
        k0 = np.where(ki >= 0, ki, 0)
        cnt = np.where(has, self.cC[p, k0], 0); s = np.where(has, self.cS[p, k0], 0)
        prof = np.where(cnt > 0, s / np.maximum(cnt, 1), np.nan)
        gc = np.where(ok, self.gC[p, plan_on.astype(int)], 0); gs = np.where(ok, self.gS[p, plan_on.astype(int)], 0)
        glob = np.where(gc > 0, gs / np.maximum(gc, 1), np.nan)
        return np.where(np.isfinite(prof), prof, glob)


def rule3_oof(F, meta, profile):
    """3단 규칙 값과 단계(0: 1주 전 사용, 1: 2주 전 사용, 2: 평균 프로파일). 프로파일은 창 기준일 이전 관측만."""
    plan_on, w1_on, w2_on = rule_inputs(F)
    n = len(meta)
    origin_day = np.repeat(pd.to_datetime(meta["origin"]).dt.floor("D").to_numpy(), 96)
    fb = profile.lookup(origin_day, plan_on.astype(np.int8), F["hour"].to_numpy(np.int8), F["minute"].to_numpy(np.int8))
    w1 = F["w1_same"].to_numpy(); w2 = F["w2_same"].to_numpy()
    step = np.where(plan_on == w1_on, 0, np.where(plan_on == w2_on, 1, 2)).astype(np.float32)
    val = np.where(step == 0, w1, np.where(step == 1, w2, fb)).astype(np.float32)
    return val, step


def rule3_official(F_tr, y_tr_long, F_eval):
    """현재 공식 3단 규칙(학습 구간 전체 평균 프로파일, 학습 정답이 섞임) — OOF와 비교용."""
    pt, _, _ = rule_inputs(F_tr)
    pf = pd.DataFrame(dict(p=pt.astype(np.int8), h=F_tr["hour"].to_numpy(np.int8), m=F_tr["minute"].to_numpy(np.int8), y=y_tr_long))
    prof = pf.groupby(["p", "h", "m"]).y.mean(); glob = pf.groupby("p").y.mean()

    def f(F):
        po, w1o, w2o = rule_inputs(F)
        keys = pd.MultiIndex.from_arrays([po.astype(np.int8), F["hour"].to_numpy(np.int8), F["minute"].to_numpy(np.int8)])
        fb = prof.reindex(keys).to_numpy(); miss = ~np.isfinite(fb)
        if miss.any(): fb[miss] = pd.Series(po[miss].astype(np.int8)).map(glob).to_numpy()
        step = np.where(po == w1o, 0, np.where(po == w2o, 1, 2)).astype(np.float32)
        return np.where(step == 0, F["w1_same"].to_numpy(), np.where(step == 1, F["w2_same"].to_numpy(), fb)).astype(np.float32), step
    return f(F_tr), f(F_eval)


# ---------------------------------------------------------------- 지표
def metrics(y, p, thr=PEAK, origin_hour=None):
    a = y >= thr; q = p >= thr
    tp = int((a & q).sum()); fp = int((~a & q).sum()); fn = int((a & ~q).sum())
    pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
    f = lambda b: (1 + b * b) * pr * rc / (b * b * pr + rc) if pr + rc else 0.0
    e = p - y
    out = dict(MAE=float(np.abs(e).mean()), RMSE=float(np.sqrt((e ** 2).mean())), bias=float(e.mean()),
               precision=pr, recall=rc, F1=f(1), F2=f(2), TP=tp, FP=fp, FN=fn, 칸=int(y.size))
    return out


def fp_per_day_00(y, p, origin_hour, thr=PEAK):
    """매일 00시 기준 24시간 예측만 모아 하루당 잘못된 경보 칸 수(비교 대상이 고르게 하루 1번 예측)."""
    sel = origin_hour == 0
    a = y[sel] >= thr; q = p[sel] >= thr
    return float((~a & q).sum() / max(sel.sum(), 1))


def tag_cells(F, meta):
    """상태 태그(예측 시점에 아는 계획 기준). 칸 단위 1차원 배열."""
    po, w1o, _ = rule_inputs(F)
    hol = F["holiday"].to_numpy() > 0
    tag = np.full(len(po), "정상 반복", dtype=object)
    tag[w1o & ~po] = "가동→정지"; tag[~w1o & po] = "정지→가동"; tag[po & hol] = "공휴일 가동"
    return tag



W_GRID = np.round(np.arange(0, 1.01, 0.1), 2)      # 고정 GRU 비중 w: pred = w*GRU + (1-w)*Q1

## 3. 상태 전환 피처 (휴무·복귀를 알려 주는 15개)
기준시각 **이전의 실적**과 **기준시각부터 23시간의 생산계획**만 쓴다: 하루 가동 여부, 전날 생산량, 연속 무생산 일수, 휴무 후 첫 가동일, 마지막 생산 후 경과 시간, 앞으로 24시간 계획 총량과 1·2주 전 비교, 공휴일 가동. 마지막 장에서 미래 값을 바꿔도 값이 변하지 않는지 검사한다.

In [ ]:
ST_COLS = ["st_day_on", "st_day_total", "st_day_known_h", "st_prev_day_total", "st_prev_day_on", "st_off_run", "st_first_op",
           "st_hrs_since_prod", "st_win_plan_total", "st_win_on_hours", "st_win_all_off", "st_win_vs_w1", "st_win_w1_total", "st_win_w2_total", "st_hol_prod"]


class HourlyIndex:
    """hourly_review.csv(시간별 원자료)에서 생산량 배열과 날짜 색인을 만든다."""
    def __init__(self, hourly):
        self.P = hourly["생산량"].to_numpy(float)
        self.row_of = {t: i for i, t in enumerate(hourly["origin"])}
        self.date_d = hourly["origin"].dt.floor("D").to_numpy()
        self.day_first = {}
        for i, d in enumerate(self.date_d):
            a, b = self.day_first.get(d, (i, i)); self.day_first[d] = (a, i)
        self.days = sorted(self.day_first); self.day_idx = {d: k for k, d in enumerate(self.days)}


def state_features(hi, origins, seq, Fcells, P=None):
    """창(기준시각)마다 상태 전환 피처를 만든다. 반환: 길이 (창 수 x 96)의 DataFrame.
    P: 시간별 생산량 배열(기본은 실제 자료). 미래 값 교란 검사에서 바꾼 배열을 넣어 볼 수 있다."""
    P = hi.P if P is None else P
    cs = np.concatenate([[0.0], np.cumsum(P)])
    DT = np.array([cs[hi.day_first[d][1] + 1] - cs[hi.day_first[d][0]] for d in hi.days])
    zrun = np.zeros(len(hi.days), int)
    for k in range(len(hi.days)):
        zrun[k] = (zrun[k - 1] + 1 if k > 0 else 1) if DT[k] == 0 else 0
    last_pos = np.full(len(P), -1); lp = -1
    for i, v in enumerate(P):
        if v > 0: lp = i
        last_pos[i] = lp
    ip = SEQ.index("planned_production"); i1 = SEQ.index("w1_production"); i2 = SEQ.index("w2_production")
    n = len(origins); out = {c: np.full((n, 96), np.nan, np.float32) for c in ST_COLS}
    for w, o in enumerate(pd.to_datetime(origins)):
        r0 = hi.row_of[o]; dA = hi.date_d[r0]; a0, a1 = hi.day_first[dA]; kA = hi.day_idx[dA]
        win = P[r0:r0 + 24]
        totA = cs[a1 + 1] - cs[a0]
        b0 = a1 + 1; b1 = min(r0 + 23, len(P) - 1)
        totB = (cs[b1 + 1] - cs[b0]) if b1 >= b0 else 0.0
        knownB = max(b1 - b0 + 1, 0)
        prevA = (cs[a0] - cs[a0 - 24]) if a0 >= 24 else np.nan
        hs = (r0 - last_pos[r0 - 1]) if r0 > 0 and last_pos[r0 - 1] >= 0 else 336
        hs = min(hs, 336)
        w1t = seq[w, :, i1].sum(); w2t = seq[w, :, i2].sum(); pt = seq[w, :, ip].sum()
        for h in range(24):
            inA = (r0 + h) <= a1
            tot, kh, prev = (totA, 24, prevA) if inA else (totB, knownB, totA)
            run = zrun[kA - 1] if inA and kA > 0 else (zrun[kA] if not inA else 0)
            sl = slice(h * 4, h * 4 + 4)
            out["st_day_on"][w, sl] = float(tot > 0); out["st_day_total"][w, sl] = tot; out["st_day_known_h"][w, sl] = kh
            out["st_prev_day_total"][w, sl] = prev; out["st_prev_day_on"][w, sl] = (float(prev > 0) if np.isfinite(prev) else np.nan)
            out["st_off_run"][w, sl] = min(run, 14); out["st_first_op"][w, sl] = float(tot > 0 and run >= 2)
            out["st_hrs_since_prod"][w, sl] = hs
            out["st_win_plan_total"][w, sl] = pt; out["st_win_on_hours"][w, sl] = float((win > 0).sum()); out["st_win_all_off"][w, sl] = float(pt == 0)
            out["st_win_vs_w1"][w, sl] = pt - w1t; out["st_win_w1_total"][w, sl] = w1t; out["st_win_w2_total"][w, sl] = w2t
    df = pd.DataFrame({c: v.reshape(-1) for c, v in out.items()})
    df["st_hol_prod"] = ((Fcells["holiday"].to_numpy() > 0) & (Fcells["planned_production"].to_numpy() > 0)).astype(np.float32)
    return df

## 4. 경보 기준 선택 함수
**실제 피크 정의(176)**와 **경보 기준(예측값에 거는 임계값)**은 다르다. `choose`는 지난 예측 결과에서 "재현율 ≥ 목표(0.85) 중 정밀도가 가장 높은 기준"을 고른다.

In [ ]:
GRID = np.arange(100.0, 231.0, 1.0)                     # 경보 기준 후보(예측값에 거는 임계값)


def m2(y, p, t, oh=None):
    """실제 피크는 y >= 176, 경보는 p >= t (t는 경보 기준)."""
    a = y >= PEAK; q = p >= t; tp = int((a & q).sum()); fp = int((~a & q).sum()); fn = int((a & ~q).sum())
    pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
    f = lambda b: (1 + b * b) * pr * rc / (b * b * pr + rc) if pr + rc else 0.0
    out = dict(MAE=float(np.abs(p - y).mean()), bias=float((p - y).mean()), precision=pr, recall=rc, F1=f(1), F2=f(2), TP=tp, FP=fp, FN=fn)
    if oh is not None: out["FP칸/일(00시)"] = float((~a & q)[oh == 0].sum() / max((oh == 0).sum(), 1))
    return out


def curve(y, p, grid=GRID):
    a = (y >= PEAK).reshape(-1); pr = p.reshape(-1); pa = np.sort(pr[a]); pn = np.sort(pr[~a])
    tp = len(pa) - np.searchsorted(pa, grid, "left"); fp = len(pn) - np.searchsorted(pn, grid, "left")
    prec = tp / np.maximum(tp + fp, 1); rec = tp / max(len(pa), 1)
    return pd.DataFrame(dict(t=grid, precision=prec, recall=rec, F2=5 * prec * rec / np.maximum(4 * prec + rec, 1e-12), FP=fp, TP=tp))


def choose(y, p, policy):
    """경보 기준 고르기. 'R85' = 재현율 ≥ .85 중 정밀도 최대(같으면 더 높은 기준), 'R90' 동일, 'F2' = F2 최대, 'T176' = 176 고정."""
    if policy == "T176": return 176.0
    cv = curve(y, p)
    if policy == "F2": return float(cv.t[int(cv.F2.to_numpy().argmax())])
    target = int(policy[1:]) / 100; ok = cv.recall >= target                    # "R85" -> 0.85, "R95" -> 0.95
    if not ok.any(): return float(cv.t.iloc[0])                       # 목표 재현율을 못 맞추면 가장 낮은 기준(재현율 최대)
    sub = cv[ok]; best = sub.precision.max(); return float(sub[sub.precision >= best - 1e-12].t.max())

## 5. 튜닝 잔차 GRU
하이퍼파라미터(은닉 32, 2층 양방향, dropout .15, batch 64, AdamW, 피크 가중치 3, 9 epoch, seed 42)는 7월 검증 튜닝(`notebooks/tuning_reference/`)에서 정해진 값을 **고정**해서 쓴다. 후보 10개의 튜닝 과정 자체는 자동 실행 대상이 아니다.

In [ ]:
import os
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

HIDDEN, LAYERS, DROPOUT, BATCH, LR, WD = 32, 2, 0.15, 64, 1e-3, 1e-4
PEAK_WEIGHT, EPOCHS, SEED = 3.0, 9, 42


class TunedGRURegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(input_size=29, hidden_size=HIDDEN, num_layers=LAYERS, batch_first=True, dropout=DROPOUT, bidirectional=True)
        self.output = nn.Linear(2 * HIDDEN, 4)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.output(out)


def _combine(sequence, recent):
    """시간별 입력 25개 + 직전 1시간 전력 4개를 24시간에 반복해 붙인다 → (n, 24, 29)."""
    rep = np.repeat(recent[:, None, :], sequence.shape[1], axis=1)
    return np.concatenate([sequence, rep], axis=2).astype(np.float32)


def train_predict(seq_tr, rec_tr, y_tr, base_tr, seq_ev, rec_ev, base_ev, peak=176.0, epochs=EPOCHS, seed=SEED, threads=None):
    """학습 창으로 학습하고 평가 창을 예측한다. 입력 y/baseline은 (n, 24, 4). 반환: (n_ev, 24, 4) 전력 예측(0 미만은 0)."""
    X_tr = _combine(seq_tr, rec_tr); X_ev = _combine(seq_ev, rec_ev)
    y_tr = y_tr.astype(np.float32); res_tr = y_tr - base_tr.astype(np.float32)
    xm = X_tr.mean(axis=(0, 1), keepdims=True); xs = X_tr.std(axis=(0, 1), keepdims=True); xs[xs < 1e-6] = 1.0
    rm = res_tr.mean(axis=(0, 1), keepdims=True); rs = res_tr.std(axis=(0, 1), keepdims=True); rs[rs < 1e-6] = 1.0
    Xs = ((X_tr - xm) / xs).astype(np.float32); Xe = ((X_ev - xm) / xs).astype(np.float32); Rs = ((res_tr - rm) / rs).astype(np.float32)
    np.random.seed(seed); torch.manual_seed(seed)
    torch.set_num_threads(threads or min(8, os.cpu_count() or 1))
    gen = torch.Generator().manual_seed(seed)
    loader = DataLoader(TensorDataset(torch.from_numpy(Xs), torch.from_numpy(Rs), torch.from_numpy(y_tr)), batch_size=BATCH, shuffle=True, generator=gen)
    model = TunedGRURegressor(); opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    hist = []
    for ep in range(1, epochs + 1):
        model.train(); tot = n = 0
        for bx, bt, ba in loader:
            opt.zero_grad(); pred = model(bx)
            w = torch.where(ba >= peak, torch.as_tensor(PEAK_WEIGHT), torch.as_tensor(1.0))
            loss = (w * (pred - bt) ** 2).mean(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0); opt.step()
            tot += loss.item() * len(bx); n += len(bx)
        hist.append(tot / n)
    model.eval()
    with torch.inference_mode():
        sp = model(torch.from_numpy(Xe)).numpy()
    return np.clip(base_ev.astype(np.float32) + sp * rs + rm, 0, None).astype(np.float32), hist

## 6. 모든 창과 피처 만들기
- **창**: 학습 가능한 창(1/15~7월 4,635개) + 8~9월 테스트 창 1,057개 = 5,692개. 평가 창은 7월 627개 + 8~9월 1,057개.
- **val 단계**: 타깃이 7/1 이전에 끝난 창(3,985개)으로 학습 → 7월 예측.  **test 단계**: 타깃이 8/1 이전에 끝난 창(4,635개)으로 학습 → 8~9월 예측.
- 3단 규칙 값·상태 전환 피처는 창마다 기준시각 이전 정보만 써서 만들므로 학습 시점과 무관하게 같은 값이다.

In [ ]:
def build_windows(prep=None, strict=True):
    """학습 가능한 창 + 테스트 창과 공통 피처 표. prep: 전처리 산출물 폴더."""
    prep = PREP if prep is None else Path(prep)
    fm_ = pd.read_csv(prep / "metadata_final_retrain_candidates.csv", parse_dates=["origin", "target_end"]); keep = as_bool(fm_["primary_eligible"])
    mt_ = pd.read_csv(prep / "metadata_test.csv", parse_dates=["origin", "target_end"])
    mv_ = pd.read_csv(prep / "metadata_validation.csv", parse_dates=["origin", "target_end"]); kv = as_bool(mv_["primary_eligible"])
    af, at, av = (np.load(prep / f"arrays_{n}.npz") for n in ("final_retrain_candidates", "test", "validation"))
    cat = lambda k: np.concatenate([af[k][keep], at[k]])
    W = types.SimpleNamespace()
    W.seq, W.rec = cat("sequence"), cat("recent")
    W.y3, W.base3 = cat("y").astype(np.float32), cat("baseline_w1").astype(np.float32)          # (N, 24, 4)
    W.N = len(W.seq); W.y = W.y3.reshape(W.N, 96); W.base = W.base3.reshape(W.N, 96); W.sus = cat("suspect_target")
    W.meta = pd.concat([fm_.loc[keep], mt_], ignore_index=True)[["origin", "target_end"]]
    idx = {o: i for i, o in enumerate(W.meta.origin)}
    W.ev_val = np.array([idx[o] for o in mv_.loc[kv, "origin"] if o in idx]); W.ev_test = np.array([idx[o] for o in mt_["origin"]])
    if strict:
        assert W.meta.origin.is_monotonic_increasing and W.N == 4635 + 1057, f"창 수 {W.N}"
        assert len(W.ev_val) == 627 and len(W.ev_test) == 1057
        assert np.array_equal(W.y[W.ev_val], av["y"][kv].reshape(-1, 96)), "검증 창 정답 불일치"
    W.EV = np.concatenate([W.ev_val, W.ev_test])
    W.hi = HourlyIndex(pd.read_csv(prep / "hourly_review.csv", parse_dates=["origin"]))
    W.F = long_features(dict(sequence=W.seq, recent=W.rec))
    st = state_features(W.hi, W.meta["origin"], W.seq, W.F)
    for c in ST_COLS: W.F[c] = st[c].to_numpy()
    prof = OOFProfile(observation_table([(dict(sequence=W.seq, y=W.y, suspect_target=W.sus), W.meta)]))
    r, s = rule3_oof(W.F, W.meta, prof)
    W.F["rule3_oof"] = r; W.F["rule_step"] = s; W.F["rule3_oof"] = W.F["rule3_oof"].fillna(W.F["w1_same"])
    W.BASE = Q1_FEATURES; W.L13 = W.BASE + ["rule3_oof", "rule_step"] + ST_COLS               # 공식 30피처 / 새 Q1 피처(47개)
    W.tr_val = (W.meta.target_end <= pd.Timestamp("2021-07-01")).to_numpy()                 # val 단계 학습 창
    W.tr_test = (W.meta.target_end <= pd.Timestamp("2021-08-01")).to_numpy()                # test 단계 학습 창
    W.mon = W.meta.origin.dt.month.to_numpy()
    return W


W = build_windows()
log(f"창 {W.N}개 | 학습 창 val {int(W.tr_val.sum())} / test {int(W.tr_test.sum())} | 평가 창 val {len(W.ev_val)} + test {len(W.ev_test)} | 피처 공식 {len(W.BASE)}개 / 새 Q1 {len(W.L13)}개")

In [ ]:
STAGES = {"val": (W.tr_val, W.ev_val, False), "test": (W.tr_test, W.ev_test, True)}      # (학습 마스크, 평가 창 위치, 예측을 0 이상으로 자를지)
GRU = {}
for stg, (trm, ev, _) in STAGES.items():
    t0 = time.time()
    p, hist = train_predict(W.seq[trm], W.rec[trm], W.y3[trm], W.base3[trm], W.seq[ev], W.rec[ev], W.base3[ev], seed=SEED)
    GRU[stg] = p.reshape(-1, 96)
    log(f"GRU({stg}) 학습 창 {int(trm.sum())} → 예측 창 {len(ev)} | 학습 {time.time() - t0:.0f}초 | 마지막 epoch 손실 {hist[-1]:.4f}")

## 7. 현재 Q1과 새 Q1
**현재 Q1**: 공식 30피처, q=0.90, 두 단계(val/test)에서 한 번씩 학습.
**새 Q1**: 47피처(공식 30 + 3단 규칙 값·단계 + 상태 전환 15), q=0.75. **월요일마다**(6/1 주~9/14) `타깃이 그 주 시작 전에 끝난 창`만으로 다시 학습해 그 주를 예측한다. **안전장치**: 연속 3평일 이상 생산이 0인 "이례적 휴무 구간"(데이터에서는 8/2~8/6)의 창은 그 구간이 끝난 뒤 재학습부터 **가중치를 0.1로 낮춘다**. 이 안전장치가 없으면 휴무 직후 복귀 주(8/9)에서 q에 따라 재현율이 .08~.92로 크게 흔들렸다(실험은 `notebooks/experiments/scripts/`). 안전장치는 이 사건을 본 뒤에 추가한 사후 설계이며, 같은 사건에서 q=0.75·0.80·0.90 세 모델이 모두 안정되는 것으로 일반성을 확인했다. 하이퍼파라미터(272트리, 학습률 .03 등)는 두 모델이 같다.

In [ ]:
def fit_lgb(cols, alpha, tr_mask, te_mask, clip, weight=None):
    prm = dict(Q1_PARAMS); prm["alpha"] = alpha; rows_tr = np.repeat(tr_mask, 96); rows_te = np.repeat(te_mask, 96)
    m_ = lgb.LGBMRegressor(**prm).fit(W.F.loc[rows_tr, cols], W.y.reshape(-1)[rows_tr], sample_weight=None if weight is None else weight[rows_tr])
    p = m_.predict(W.F.loc[rows_te, cols]).reshape(-1, 96).astype(np.float32)
    return np.clip(p, 0, None) if clip else p


Q1_CUR = {}
for stg, (trm, ev, clip) in STAGES.items():
    te = np.zeros(W.N, bool); te[ev] = True; t0 = time.time()
    Q1_CUR[stg] = fit_lgb(W.BASE, 0.90, trm, te, clip); log(f"현재 Q1({stg}, 공식 30피처, q=0.90) {time.time() - t0:.0f}초")

WEEKS = pd.date_range("2021-05-31", "2021-09-13", freq="7D")
NEW_Q = 0.75
# 이례적 휴무 구간(연속 3평일 이상 생산 0)을 찾는다. 재학습 때 이미 끝난 구간의 창은 가중치 0.1로 낮춘다(안전장치).
_dp = pd.Series(W.hi.P).groupby(W.hi.date_d).sum(); _off = (_dp <= 0) & (pd.DatetimeIndex(_dp.index).dayofweek < 5); STREAKS = []; _run = []
for _d, _o in _off.items():
    if _o: _run.append(_d)
    else:
        if len(_run) >= 3: STREAKS.append((_run[0], _run[-1]))
        _run = []
if len(_run) >= 3: STREAKS.append((_run[0], _run[-1]))
log(f"이례적 휴무 구간(연속 3평일 이상 생산 0): {[(str(a.date()), str(b.date())) for a, b in STREAKS]}")
Q1_NEW = np.full((W.N, 96), np.nan, np.float32); t0 = time.time(); schedule = []
for S in WEEKS:
    tr_m = (W.meta.target_end <= S).to_numpy(); te_m = ((W.meta.origin >= S) & (W.meta.origin < S + pd.Timedelta(days=7))).to_numpy()
    if te_m.sum() == 0: continue
    assert W.meta.target_end[tr_m].max() <= S, "학습 창이 주 시작 이후에 끝나는 타깃을 포함"       # 미래 정보 금지 확인
    known = np.zeros(W.N, bool)                                                                   # 재학습 시점에 이미 끝난 휴무 구간과 겹치는 창
    for a, b in STREAKS:
        if b + pd.Timedelta(days=1) <= S: known |= ((W.meta.origin < b + pd.Timedelta(days=1)) & (W.meta.target_end > a)).to_numpy()
    Q1_NEW[te_m] = fit_lgb(W.L13, NEW_Q, tr_m, te_m, True, weight=np.repeat(np.where(known, 0.1, 1.0), 96))
    schedule.append(dict(주시작=S.date(), 학습창=int(tr_m.sum()), 예측창=int(te_m.sum()), 가중치낮춘창=int((known & tr_m).sum())))
log(f"새 Q1(L1+L3, q={NEW_Q}) 주간 재학습 {len(schedule)}주 {time.time() - t0:.0f}초")
display(pd.DataFrame(schedule).T)

## 8. 경보 기준 갱신 (새 Q1)
각 평가 주의 경보 기준은 **그 주가 시작되기 전에 타깃이 끝난 최근 4주**의 (예측, 정답)에서 "재현율 ≥ .85 중 정밀도 최대"로 고른다. 이력이 없으면 176. 8~9월 결과는 기준 선택에 쓰이지 않는다.

In [ ]:
WK_OF = ((W.meta.origin - WEEKS[0]) // pd.Timedelta(days=7)).to_numpy()


HIST_LOG = []                                    # (주 시작, 이력으로 쓴 창의 가장 늦은 타깃 종료 시각) — 자체 검증에서 확인


def window_thresholds(pred, policy="R85-최근4주", target="R85"):
    """모든 창에 대한 경보 기준 벡터. 그 주 시작 전에 타깃이 끝난 창의 (예측, 정답)만 이력으로 쓴다."""
    thr = np.full(W.N, 176.0)
    if policy == "T176": return thr
    for w in np.unique(WK_OF[W.EV]):
        S = WEEKS[0] + pd.Timedelta(days=7) * int(w)
        hist = np.isfinite(pred[:, 0]) & (W.meta.target_end <= S).to_numpy() & (W.meta.origin >= WEEKS[0]).to_numpy()
        if policy == "R85-최근4주": hist &= (W.meta.origin >= S - pd.Timedelta(days=28)).to_numpy()
        if hist.sum() > 0:
            thr[WK_OF == w] = choose(W.y[hist], pred[hist], target); HIST_LOG.append((S, W.meta.target_end[hist].max()))
    return thr


THR_NEW = window_thresholds(Q1_NEW)
_wk = pd.DataFrame({"주시작": [(WEEKS[0] + pd.Timedelta(days=7) * int(w)).date() for w in np.unique(WK_OF[W.EV])],
                    "경보기준": [THR_NEW[WK_OF == w][0] for w in np.unique(WK_OF[W.EV])]})
display(_wk.set_index("주시작").T)

## 8-1. 곡선(수치 예측) 출력: 3단 규칙 + 잔차, 휴무일은 규칙
**목적별 출력 분리**: 피크 경보에는 상위 분위수(새 Q1)가 좋지만, 수치 예측(곡선)에는 상위 분위수가 높게 편향되어 오차가 크다. 그래서 곡선은 따로 만든다.
- **3단 규칙 값**을 기준으로 두고, 그 위에 LightGBM(MAE를 줄이는 L1 목적)이 **잔차(실제 − 규칙)**를 학습한다. 월요일마다 재학습. 학습에는 **가동일 칸만** 쓴다(휴무일 칸은 아래 프로파일로 대체되므로. `notebooks/experiments/19_curve_onday_training.ipynb`에서 모든 칸으로 학습한 경우보다 7월+8~9월 MAE가 0.66 낮고 주 단위 95% 구간이 0보다 작음을 확인한 뒤 채택)
- **휴무일(목표 날에 생산계획이 없는 칸)은 "과거 휴무일의 시각별 중앙값 프로파일"을 쓴다.** 학습 모델은 학습에서 본 적 없는 상황(예: 8/2 휴무 주)에서 흔들린다. 3단 규칙도 시간 단위로 가동 여부가 1주 전과 같으면 1주 전 값을 복사하는데, 휴무 주 저녁에는 1주 전(평일) 저녁도 "생산 없음"이라 같다고 보고 평일 저녁의 높은 전력을 복사한다. 그래서 휴무일에는 **기준일 이전의 휴무일들의 (시각, 15분)별 중앙값**을 쓴다. 기준시각에 이미 아는 정보(생산계획, 과거 실적)만 쓴다. 이 설계는 8/2 휴무 주의 실패를 본 뒤에 추가한 **사후 설계**이며 7월(휴무일 18%)에서도 같은 방향의 개선을 확인했다(`notebooks/experiments/15_q1_curve_separation.ipynb`).

In [ ]:
RULE = W.F["rule3_oof"].to_numpy().reshape(W.N, 96).astype(np.float32)
GATE = (W.F["st_day_on"].to_numpy().reshape(W.N, 96) == 0)                  # 목표 날 생산계획 없음(휴무일)
ON_ROWS = (~GATE).reshape(-1)                                                  # 잔차 모델 학습에 쓰는 칸 = 가동일 칸
RES_TARGET = (W.y - RULE).reshape(-1).astype(np.float64); CURVE_RES = np.full((W.N, 96), np.nan, np.float32); t0 = time.time()
for S in WEEKS:
    tr_m = (W.meta.target_end <= S).to_numpy(); te_m = ((W.meta.origin >= S) & (W.meta.origin < S + pd.Timedelta(days=7))).to_numpy()
    if te_m.sum() == 0: continue
    prm = dict(Q1_PARAMS); prm["objective"] = "regression_l1"; rows_tr = np.repeat(tr_m, 96) & ON_ROWS; rows_te = np.repeat(te_m, 96)      # 학습은 가동일 칸만(휴무일 칸은 아래에서 프로파일을 쓰므로 모델이 배울 필요가 없다)
    m_ = lgb.LGBMRegressor(**prm).fit(W.F.loc[rows_tr, W.L13], RES_TARGET[rows_tr])
    CURVE_RES[te_m] = np.clip(m_.predict(W.F.loc[rows_te, W.L13]).reshape(-1, 96) + RULE[te_m], 0, None)
HRLY = pd.read_csv(PREP / "hourly_review.csv", parse_dates=["origin"]).copy(); HRLY["date"] = HRLY.origin.dt.floor("D"); HRLY["hr"] = HRLY.origin.dt.hour
_PW = ["15분", "30분", "45분", "60분"]; _dp = HRLY.groupby("date")["생산량"].sum(); _off = _dp[_dp == 0].index                # 하루 생산이 0인 날(휴무일)
_byday = {d: g.sort_values("hr")[_PW].to_numpy() for d, g in HRLY.groupby("date") if len(g) == 24}
IDLE = np.full((W.N, 96), np.nan, np.float32); _cache = {}; dates_ = W.meta.origin.dt.floor("D").to_numpy(); hr0_ = W.meta.origin.dt.hour.to_numpy()
for i in range(W.N):
    D = pd.Timestamp(dates_[i])
    if D not in _cache:
        ds = [d for d in _off if d < D and d in _byday]                                    # 기준일 이전의 휴무일만 사용(미래 정보 없음)
        _cache[D] = np.median(np.stack([_byday[d] for d in ds]), axis=0) if len(ds) >= 3 else None
    if _cache[D] is not None: IDLE[i] = _cache[D][(hr0_[i] + np.repeat(np.arange(24), 4)) % 24, np.tile(np.arange(4), 24)]
CURVE = np.where(GATE & np.isfinite(IDLE), IDLE, np.where(GATE, RULE, CURVE_RES)).astype(np.float32)
log(f"곡선(3단 규칙 + 잔차 L1, 휴무일은 프로파일) 주간 재학습 {time.time() - t0:.0f}초 | 휴무일 칸 비율 7월 {GATE[W.ev_val].mean():.3f} / 8~9월 {GATE[W.ev_test].mean():.3f}")

## 8-2. 2단계 경보와 피크 확률
- **경보**: 새 Q1 예측 ≥ 경보 기준(재현율 목표 .85, 최근 4주 이력으로 주마다 갱신). **주의**: 같은 방식으로 재현율 목표 .95의 더 낮은 기준. 주의는 "경보가 놓친 경계 근처 피크"를 가볍게 대비하는 단계다.
- **피크 확률**: 새 Q1의 예측값을 `P(전력 ≥ 176)`로 바꾸는 **로지스틱(절편 + 기울기) 보정**을, 각 주마다 그 주 시작 전에 타깃이 끝난 **최근 4주** 결과로 학습한다(미래 정보 없음). isotonic·분류기 등 다른 방법은 8~9월에 맞지 않아 쓰지 않았다(`notebooks/experiments/scripts/`).

In [ ]:
from sklearn.linear_model import LogisticRegression
THR_WATCH = window_thresholds(Q1_NEW, target="R95")
LAB = (W.y >= PEAK); PROB = np.full((W.N, 96), np.nan, np.float32)
for w in np.unique(WK_OF[W.EV]):
    S = WEEKS[0] + pd.Timedelta(days=7) * int(w); wk_ = WK_OF == w
    hist = np.isfinite(Q1_NEW[:, 0]) & (W.meta.target_end <= S).to_numpy() & (W.meta.origin >= WEEKS[0]).to_numpy() & (W.meta.origin >= S - pd.Timedelta(days=28)).to_numpy()
    lr = LogisticRegression(C=1e4, max_iter=200).fit((Q1_NEW[hist].reshape(-1, 1) - 176.0) / 20.0, LAB[hist].reshape(-1))
    PROB[wk_] = lr.predict_proba((Q1_NEW[wk_].reshape(-1, 1) - 176.0) / 20.0)[:, 1].reshape(-1, 96)
log("2단계 경보 기준(주의)와 피크 확률(Platt, 최근 4주) 계산 완료")

## 9. 결과 지표와 예측 파일 저장
같은 평가 칸에서 아래 모델을 비교한다(모두 8~9월은 개발 검증).
`baseline_w1` 1주 전 기준선 / `q1_current` 현재 Q1 / `gru_tuned` 튜닝 GRU / `ens_current` 현재 앙상블(GRU .5 + 현재 Q1 .5) / `q1_new` 새 Q1 / `ens_new_w05`, `ens_new_w02` 새 Q1과 GRU의 결합(GRU 비중 0.5, 0.2). `@176`은 경보 기준 176 고정, `@R85-최근4주`는 주별 갱신 기준.

In [ ]:
def summary_metrics(y_, p_, t):
    """실제 피크 y>=176, 경보 p>=t(창별 경보 기준). F1·F2·정확도와 경보를 안 울릴 때의 정확도(함정)를 함께 낸다."""
    t = np.broadcast_to(np.asarray(t, float).reshape(-1, 1), y_.shape) if np.ndim(t) else t
    a = y_ >= 176.0; q = p_ >= t; tp = int((a & q).sum()); fp = int((~a & q).sum()); fn = int((a & ~q).sum()); tn = int((~a & ~q).sum())
    pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
    f = lambda b: (1 + b * b) * pr * rc / (b * b * pr + rc) if pr + rc else 0.0
    e = p_ - y_
    return dict(MAE=float(np.abs(e).mean()), RMSE=float(np.sqrt((e ** 2).mean())), bias=float(e.mean()), precision=pr, recall=rc, F1=f(1), F2=f(2),
                accuracy=(tp + tn) / y_.size, no_alarm_accuracy=float((~a).mean()), peak_rate=float(a.mean()), TP=tp, FP=fp, FN=fn, TN=tn, cells=int(y_.size))


POS = {"val": W.ev_val, "test": W.ev_test}; PRED = {}
for stg in ("val", "test"):
    ev = POS[stg]
    PRED[stg] = dict(baseline_w1=W.base[ev], gru_tuned=GRU[stg], q1_current=Q1_CUR[stg], q1_new=Q1_NEW[ev])
    PRED[stg]["ens_current"] = 0.5 * GRU[stg] + 0.5 * Q1_CUR[stg]
    PRED[stg]["ens_new_w05"] = 0.5 * GRU[stg] + 0.5 * Q1_NEW[ev]
    PRED[stg]["ens_new_w02"] = 0.2 * GRU[stg] + 0.8 * Q1_NEW[ev]
    PRED[stg]["rule3"] = RULE[ev]; PRED[stg]["curve_final"] = CURVE[ev]; PRED[stg]["prob_peak"] = PROB[ev]
SPECS = [("baseline_w1@176", "baseline_w1", False), ("q1_current@176", "q1_current", False), ("gru_tuned@176", "gru_tuned", False),
         ("ens_current@176", "ens_current", False), ("q1_new@176", "q1_new", False), ("q1_new@R85-최근4주", "q1_new", True),
         ("ens_new_w05@176", "ens_new_w05", False), ("ens_new_w02@176", "ens_new_w02", False)]
rows = []
for stg, label in (("val", "7월(검증)"), ("test", "8~9월(개발 검증)")):
    for name, key, adaptive in SPECS:
        t = THR_NEW[POS[stg]] if adaptive else 176.0
        rows.append(dict(구간=label, 모델=name, **summary_metrics(W.y[POS[stg]], PRED[stg][key], t)))
METRICS = pd.DataFrame(rows); METRICS.to_csv(OUTDIR / "metrics_summary.csv", index=False, encoding="utf-8-sig")
show = ["MAE", "bias", "precision", "recall", "F1", "F2", "accuracy", "no_alarm_accuracy", "FP", "FN"]
display(METRICS[METRICS.구간 == "7월(검증)"].set_index("모델")[show].round(3).rename_axis("7월(검증)"))
display(METRICS[METRICS.구간 != "7월(검증)"].set_index("모델")[show].round(3).rename_axis("8~9월(개발 검증)"))

# ---- 곡선(수치 예측) 지표
rows = []
for stg, label in (("val", "7월(검증)"), ("test", "8~9월(개발 검증)")):
    yy = W.y[POS[stg]]
    for name, key in (("baseline_w1", "baseline_w1"), ("rule3(3단 규칙 값)", "rule3"), ("gru_tuned", "gru_tuned"), ("ens_current", "ens_current"), ("ens_new_w05", "ens_new_w05"), ("curve_final(규칙+잔차, 휴무일은 프로파일)", "curve_final")):
        e = PRED[stg][key] - yy; rows.append(dict(구간=label, 모델=name, MAE=float(np.abs(e).mean()), RMSE=float(np.sqrt((e ** 2).mean())), bias=float(e.mean())))
CURVE_M = pd.DataFrame(rows); CURVE_M.to_csv(OUTDIR / "metrics_curve.csv", index=False, encoding="utf-8-sig")
display(CURVE_M.pivot(index="모델", columns="구간", values=["MAE", "RMSE", "bias"]).round(2))

# ---- 2단계 경보 지표
rows = []
for stg, label in (("val", "7월(검증)"), ("test", "8~9월(개발 검증)")):
    ev = POS[stg]; yy = W.y[ev]; pr = PRED[stg]["q1_new"]; ta = THR_NEW[ev][:, None]; tw = THR_WATCH[ev][:, None]
    a = yy >= 176; A = pr >= ta; Wt = (pr >= tw) & ~A; AW = A | Wt; prec = lambda m_: float((a & m_).sum() / max(m_.sum(), 1))
    rows.append(dict(구간=label, 경보_정밀도=prec(A), 경보_재현율=float((a & A).sum() / a.sum()), 경보_칸비율=float(A.mean() * 100), 주의만_칸비율=float(Wt.mean() * 100), 주의만_정밀도=prec(Wt),
                     경보와주의_재현율=float((a & AW).sum() / a.sum()), 경보와주의_정밀도=prec(AW), 경보가놓친피크_중_주의가잡은_비율=float((a & ~A & Wt).sum() / max((a & ~A).sum(), 1) * 100), 놓친피크_경보=int((a & ~A).sum()), 놓친피크_경보와주의=int((a & ~AW).sum())))
LEVELS = pd.DataFrame(rows); LEVELS.to_csv(OUTDIR / "metrics_alarm_levels.csv", index=False, encoding="utf-8-sig"); display(LEVELS.set_index("구간").T.round(3))

# ---- 확률 품질
def ap_score(a_, s_):
    o = np.argsort(-s_, kind="stable"); a_o = a_[o]; tp = np.cumsum(a_o); fp = np.cumsum(~a_o); prec_ = tp / (tp + fp); rec_ = tp / max(a_.sum(), 1)
    return float(np.sum(np.diff(np.concatenate([[0], rec_])) * prec_))
rows = []
for stg, label in (("val", "7월(검증)"), ("test", "8~9월(개발 검증)")):
    a = (W.y[POS[stg]] >= 176).reshape(-1); s_ = np.clip(PRED[stg]["prob_peak"].reshape(-1).astype(float), 1e-6, 1 - 1e-6); o = np.argsort(s_); ece = 0.0
    for ch in np.array_split(o, 10): ece += len(ch) / len(s_) * abs(s_[ch].mean() - a[ch].mean())
    rows.append(dict(구간=label, Brier=float(np.mean((s_ - a) ** 2)), LogLoss=float(-np.mean(a * np.log(s_) + (~a) * np.log(1 - s_))), ECE=float(ece), AP=ap_score(a, s_), 평균_예측확률=float(s_.mean()), 실제_피크비율=float(a.mean())))
PROBM = pd.DataFrame(rows); PROBM.to_csv(OUTDIR / "metrics_probability.csv", index=False, encoding="utf-8-sig"); display(PROBM.set_index("구간").T.round(4))

for stg, fname in (("test", "test_predictions_8_9월.csv"), ("val", "validation_predictions_7월.csv")):
    ev = POS[stg]; n = len(ev); origin = np.repeat(W.meta.origin.to_numpy()[ev], 96); step = np.tile(np.arange(96), n)
    ttime = pd.to_datetime(origin) + pd.to_timedelta(step // 4, "h") + pd.to_timedelta(15 * (step % 4 + 1), "m")
    thr = np.repeat(THR_NEW[ev], 96)
    df = pd.DataFrame(dict(기준시각=origin, 예측대상시각=ttime, 예측칸=step, 실제전력=W.y[ev].reshape(-1)))
    for k, v in PRED[stg].items(): df[k] = v.reshape(-1).round(3)
    df["최종_경보기준"] = 176.0
    df["최종_피크경보"] = (df["q1_new"] >= 176.0).astype(int)
    df["새Q1_경보기준(주별)"] = thr; df["실제피크(>=176)"] = (df["실제전력"] >= 176).astype(int)
    df["경보_새Q1(주별 기준)"] = (df["q1_new"] >= thr).astype(int); df["경보_현재Q1(176)"] = (df["q1_current"] >= 176).astype(int)
    thr_w = np.repeat(THR_WATCH[ev], 96); df["주의기준(주별)"] = thr_w
    df["경보단계(0정상/1주의/2경보)"] = np.where(df["q1_new"] >= thr, 2, np.where(df["q1_new"] >= thr_w, 1, 0))
    df.to_csv(OUTDIR / fname, index=False, encoding="utf-8-sig"); log(f"저장: outputs/{fname} ({len(df):,}행)")

### 9-1. 그림
왼쪽: 8~9월(개발 검증) 모델별 피크 F2와 MAE. 오른쪽: 새 Q1의 주별 경보 기준과 그 주의 실제 재현율(목표 .85).

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
d = METRICS[METRICS.구간 != "7월(검증)"].set_index("모델"); nm = [s[0] for s in SPECS]
ax[0].bar(range(len(nm)), d.loc[nm, "F2"], color="#0f766e"); ax[0].set_xticks(range(len(nm))); ax[0].set_xticklabels(nm, rotation=40, ha="right", fontsize=8); ax[0].set_ylim(0, 1); ax[0].set_title("8~9월(개발 검증) 피크 F2")
ax[1].bar(range(len(nm)), d.loc[nm, "MAE"], color="#d97706"); ax[1].set_xticks(range(len(nm))); ax[1].set_xticklabels(nm, rotation=40, ha="right", fontsize=8); ax[1].set_title("8~9월(개발 검증) MAE (낮을수록 좋음)")
ws = np.unique(WK_OF[W.EV]); xs, rs, th = [], [], []
for w in ws:
    k = WK_OF[W.EV] == w; yy = W.y[W.EV][k]; pp_ = Q1_NEW[W.EV][k]; tt = THR_NEW[W.EV][k]; a_ = yy >= 176
    xs.append(WEEKS[0] + pd.Timedelta(days=7) * int(w)); th.append(tt[0]); rs.append(((a_ & (pp_ >= tt[:, None])).sum() / a_.sum()) if a_.sum() >= 50 else np.nan)
ax[2].plot(xs, rs, marker="o", color="#0f766e", label="주별 실제 재현율"); ax[2].axhline(0.85, color="k", ls=":"); ax[2].set_ylabel("재현율"); a2 = ax[2].twinx(); a2.step(xs, th, where="post", color="#d97706", label="경보 기준")
a2.set_ylabel("경보 기준"); ax[2].set_title("새 Q1: 주별 경보 기준과 재현율"); ax[2].tick_params(axis="x", rotation=30)
plt.tight_layout(); fig.savefig(OUTDIR / "figures/summary.png", dpi=130); plt.show()

## 10. 자체 검증
1. **기준 수치와 대조**: 실험 때 저장해 둔 결과(`reference/expected_metrics.csv`)와 이번 실행 결과가 허용 오차 안에서 같은지 확인한다(GRU가 들어간 모델은 허용 오차를 더 넓게 둔다: 8~9월 GRU는 최종 재학습 방식 그대로라 정확히 재현되지만, 7월 GRU는 원래 튜닝 노트북에서 여러 후보를 이어서 학습한 경로로 만들어져 처음부터 학습한 결과와 소폭 다르다).
2. **구조 점검**: 창 개수, 학습 창이 평가 시작 전에 끝나는지.
3. **미래 값 교란 검사(누수 점검)**
   - (a) 기준시각+24시간 **이후의 생산량**을 엉터리로 바꿔도 상태 전환 피처가 변하지 않는가.
   - (b) 2021-08-15 이후의 **전력 원자료**를 엉터리로 바꾸고 **전처리부터 다시 실행**해, 8/15 이전에 기준시각이 있는 모든 창의 피처(공식 30 + 3단 규칙 값·단계 + 상태 전환 15)가 그대로인가. 8/15 이후 창은 바뀌어야 한다(검사가 실제로 작동하는지 확인).

In [ ]:
PREPROCESS_SRC = r'''
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print  # Jupyter 없이 스크립트로 검증할 때 사용

ROOT = Path.cwd()
if not (ROOT / 'data/okm_augumented_2021.csv').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data/okm_augumented_2021.csv'
OUT = ROOT / 'results/preprocessing_24h'
POWER_COLS = ['15분', '30분', '45분', '60분']
MINUTES = [15, 30, 45, 60]
HORIZON = 24
VALID_START = pd.Timestamp('2021-07-01')
TEST_START = pd.Timestamp('2021-08-01')
SUSPECT_DATES = pd.to_datetime(['2021-07-13', '2021-07-15'])
# 기존 전처리에서 사용한 대한민국 2021년 공휴일 달력을 유지합니다.
HOLIDAYS = pd.to_datetime([
    '2021-01-01', '2021-02-11', '2021-02-12', '2021-02-13',
    '2021-03-01', '2021-05-05', '2021-05-19', '2021-06-06',
    '2021-08-15', '2021-08-16', '2021-09-20', '2021-09-21',
    '2021-09-22', '2021-10-03', '2021-10-04', '2021-10-09',
    '2021-10-11', '2021-12-25'
])
raw = pd.read_csv(DATA_PATH)
print('원본:', raw.shape)

data = raw.copy()
data['date'] = pd.to_datetime(data['날짜'].astype(str), format='%Y%m%d')
data['original_hour'] = data['시간']
row_hour = data.groupby('date', sort=False).cumcount()
valid_hour = data['시간'].between(0, 23) & data['시간'].eq(data['시간'].round())
assert data.groupby('date').size().eq(24).all(), '날짜별 행 수 확인 필요'
assert data.loc[valid_hour, '시간'].eq(row_hour[valid_hour]).all(), '원본 행 순서 확인 필요'
data['time_repaired'] = ~valid_hour
data.loc[~valid_hour, '시간'] = row_hour[~valid_hour]
data['origin'] = data['date'] + pd.to_timedelta(data['시간'], unit='h')
data = data.sort_values('origin').reset_index(drop=True)
assert data['origin'].is_unique
assert data['origin'].diff().dropna().eq(pd.Timedelta(hours=1)).all()
assert np.isfinite(data[POWER_COLS + ['생산량']].to_numpy(dtype=float)).all()
assert data[POWER_COLS + ['생산량']].ge(0).all().all()
data['suspect_date'] = data['date'].isin(SUSPECT_DATES)
# 11시 행 전체를 포함: 행 시작시각 기준 [8/28 18시, 8/29 12시).
data['special_period'] = data['origin'].ge('2021-08-28 18:00') & data['origin'].lt('2021-08-29 12:00')
calendar = pd.DataFrame({
    'month': data['origin'].dt.month,
    'day': data['origin'].dt.day,
    'hour': data['origin'].dt.hour,
    'weekday': data['origin'].dt.dayofweek,
    'weekend': data['origin'].dt.dayofweek.ge(5).astype(int),
    'holiday': data['date'].isin(HOLIDAYS).astype(int),
})
display(data.loc[data['time_repaired'], ['date', 'original_hour', '시간']].head())
print('복구 행:', int(data['time_repaired'].sum()))
print('범위:', data['origin'].min(), '~', data['origin'].max())

# 시간축의 연속성을 검증했으므로 정수 위치로 정확한 시간 블록을 조회합니다.
starts = np.arange(336, len(data) - HORIZON + 1)
target_pos = starts[:, None] + np.arange(HORIZON)
w1_pos, w2_pos = target_pos - 168, target_pos - 336
power = data[POWER_COLS].to_numpy(dtype=float)
production = data['생산량'].to_numpy(dtype=float)
holiday = calendar['holiday'].to_numpy()
plan = production[target_pos]
sequence_parts = [plan[..., None]]
sequence_names = ['planned_production']
for name in calendar.columns:
    sequence_parts.append(calendar[name].to_numpy()[target_pos][..., None])
    sequence_names.append(name)
for week, pos in [(1, w1_pos), (2, w2_pos)]:
    sequence_parts.extend([
        power[pos], production[pos][..., None], holiday[pos][..., None],
        (plan - production[pos])[..., None],
        ((plan > 0) != (production[pos] > 0))[..., None],
        (holiday[target_pos] != holiday[pos])[..., None],
    ])
    sequence_names.extend([
        *[f'w{week}_power_{m}' for m in MINUTES],
        f'w{week}_production', f'w{week}_holiday',
        f'w{week}_production_difference', f'w{week}_production_state_mismatch',
        f'w{week}_holiday_mismatch',
    ])
sequence = np.concatenate(sequence_parts, axis=2).astype(float)
recent = power[starts - 1]
targets = power[target_pos]
recent_names = [f'recent_previous_hour_{m}' for m in MINUTES]
feature_names = recent_names + [f'h{h:02d}_{name}' for h in range(HORIZON) for name in sequence_names]
target_names = [f'h{h:02d}_power_{m}' for h in range(HORIZON) for m in MINUTES]
sample_ids = pd.Index(starts, name='sample_id')
X = pd.DataFrame(np.concatenate([recent, sequence.reshape(len(starts), -1)], axis=1), index=sample_ids, columns=feature_names)
y = pd.DataFrame(targets.reshape(len(starts), -1), index=sample_ids, columns=target_names)
print('X:', X.shape, '| y:', y.shape, '| sequence:', sequence.shape)

meta = pd.DataFrame(index=sample_ids)
meta['origin'] = data['origin'].iloc[starts].to_numpy()
meta['target_end'] = meta['origin'] + pd.Timedelta(hours=HORIZON)
meta['origin_hour'] = meta['origin'].dt.hour
suspect = data['suspect_date'].to_numpy()
special = data['special_period'].to_numpy()
meta['target_suspect'] = suspect[target_pos].any(axis=1)
meta['recent_suspect'] = suspect[starts - 1]
meta['w1_suspect'] = suspect[w1_pos].any(axis=1)
meta['w2_suspect'] = suspect[w2_pos].any(axis=1)
meta['primary_eligible'] = ~meta['target_suspect']
meta['history_clean_eligible'] = ~meta[['target_suspect', 'recent_suspect', 'w1_suspect', 'w2_suspect']].any(axis=1)
meta['special_period_overlap'] = special[target_pos].any(axis=1)
meta['split'] = 'boundary_excluded'
meta.loc[meta['target_end'].le(VALID_START), 'split'] = 'train'
meta.loc[meta['origin'].ge(VALID_START) & meta['target_end'].le(TEST_START), 'split'] = 'validation'
meta.loc[meta['origin'].ge(TEST_START), 'split'] = 'test'
meta['final_retrain_candidate'] = meta['target_end'].le(TEST_START)
split_masks = {name: meta['split'].eq(name).to_numpy() for name in ['train', 'validation', 'test']}
split_masks['final_retrain_candidates'] = meta['final_retrain_candidate'].to_numpy()

# 기준은 겹친 윈도우가 아니라 1~6월 원본의 각 관측값을 딱 한 번 사용합니다.
threshold_source = data.loc[data['origin'].lt(VALID_START), POWER_COLS].to_numpy().ravel()
peak_threshold = float(np.quantile(threshold_source, 0.95))
hour_masks = {
    'suspect_target': suspect[target_pos],
    'special_period': special[target_pos],
    'weekend_production': (calendar['weekend'].to_numpy()[target_pos] == 1) & (plan > 0),
    'holiday_production': (holiday[target_pos] == 1) & (plan > 0),
    'zero_production': plan == 0,
    'production_both_weeks_zero': (plan > 0) & (production[w1_pos] == 0) & (production[w2_pos] == 0),
}
target_masks = {name: np.repeat(values, 4, axis=1) for name, values in hour_masks.items()}
for name, lo, hi in [('horizon_0_3', 0, 3), ('horizon_3_12', 3, 12), ('horizon_12_24', 12, 24)]:
    hours = (np.arange(24) >= lo) & (np.arange(24) < hi)
    target_masks[name] = np.broadcast_to(np.repeat(hours, 4), y.shape).copy()
target_masks['actual_peak'] = y.to_numpy() >= peak_threshold
split_summary = meta.groupby('split').agg(samples=('origin', 'size'), first_origin=('origin', 'min'), last_origin=('origin', 'max'), primary_samples=('primary_eligible', 'sum'), history_clean_samples=('history_clean_eligible', 'sum'))
display(split_summary)
print('고정 피크 기준(q95):', peak_threshold)

assert X.index.equals(y.index) and y.index.equals(meta.index)
assert np.isfinite(X.to_numpy()).all() and np.isfinite(y.to_numpy()).all()
assert X.columns.is_unique and y.columns.is_unique
assert meta['origin'].iloc[0] == pd.Timestamp('2021-01-15')
assert meta.loc[meta['split'].eq('train'), 'origin'].max() == pd.Timestamp('2021-06-30')
for pos in [w1_pos, w2_pos]:
    latest_observed = data['origin'].to_numpy()[pos[:, -1]] + np.timedelta64(60, 'm')
    assert (latest_observed <= meta['origin'].to_numpy()).all()
assert (data['origin'].to_numpy()[starts - 1] + np.timedelta64(60, 'm') == meta['origin'].to_numpy()).all()
lookup = data.set_index('origin')
for i in [0, len(starts) // 2, len(starts) - 1]:
    origin = meta['origin'].iloc[i]
    times = pd.date_range(origin, periods=24, freq='h')
    np.testing.assert_array_equal(targets[i], lookup.loc[times, POWER_COLS].to_numpy())
    np.testing.assert_array_equal(recent[i], lookup.loc[origin - pd.Timedelta(hours=1), POWER_COLS].to_numpy(dtype=float))
    for week in [1, 2]:
        cols = [sequence_names.index(f'w{week}_power_{m}') for m in MINUTES]
        np.testing.assert_array_equal(sequence[i][:, cols], lookup.loc[times - pd.Timedelta(weeks=week), POWER_COLS].to_numpy())
for name, mask in split_masks.items():
    assert mask.any(), f'{name}: 빈 분할'
assert meta.loc[split_masks['train'], 'target_end'].le(VALID_START).all()
assert meta.loc[split_masks['validation'], 'origin'].ge(VALID_START).all()
assert meta.loc[split_masks['validation'], 'target_end'].le(TEST_START).all()
assert meta.loc[split_masks['test'], 'origin'].ge(TEST_START).all()
assert meta['target_end'].max() == data['origin'].max() + pd.Timedelta(hours=1)
assert len(threshold_source) == 181 * 24 * 4
np.testing.assert_array_equal(X.iloc[:, 4:].to_numpy().reshape(sequence.shape), sequence)
print('원본 대응·과거 정보 시각·정답 경계·배열 순서 검증 통과')

OUT.mkdir(parents=True, exist_ok=True)
for name, mask in split_masks.items():
    for prefix, frame in [('X', X), ('y', y), ('metadata', meta)]:
        frame.loc[mask].to_csv(OUT / f'{prefix}_{name}.csv', encoding='utf-8-sig', index=True)
    np.savez_compressed(
        OUT / f'arrays_{name}.npz',
        sample_id=sample_ids.to_numpy()[mask], sequence=sequence[mask], recent=recent[mask],
        y=targets[mask], baseline_w1=power[w1_pos][mask], baseline_w2=power[w2_pos][mask],
        baseline_weekly_mean=((power[w1_pos] + power[w2_pos]) / 2)[mask],
        **{key: value[mask] for key, value in target_masks.items()},
    )
meta.to_csv(OUT / 'all_sample_metadata.csv', encoding='utf-8-sig')
split_summary.to_csv(OUT / 'split_summary.csv', encoding='utf-8-sig')
data.to_csv(OUT / 'hourly_review.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'feature': feature_names}).to_csv(OUT / 'feature_catalog.csv', index=False, encoding='utf-8-sig')
pd.DataFrame([
    {'target': f'h{h:02d}_power_{m}', 'hour_offset': h, 'minute_in_hour': m, 'observation_offset_minutes': h * 60 + m}
    for h in range(24) for m in MINUTES
]).to_csv(OUT / 'target_catalog.csv', index=False, encoding='utf-8-sig')
config = {
    'source_file': str(DATA_PATH.relative_to(ROOT)),
    'source_sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    'horizon_hours': 24, 'stride_hours': 1, 'history_hours': [168, 336],
    'feature_columns': feature_names, 'sequence_features': sequence_names,
    'recent_features': recent_names, 'target_columns': target_names,
    'flatten_order': 'hour first, then feature / power minute (15,30,45,60)',
    'peak_threshold': peak_threshold, 'peak_comparison': '>=',
    'threshold_source': 'unique raw observations Jan 1 through June 30; fixed after retraining',
    'threshold_observations': len(threshold_source),
    'validation_start': str(VALID_START), 'test_start': str(TEST_START),
    'holiday_dates': [str(d.date()) for d in HOLIDAYS],
    'suspect_dates': [str(d.date()) for d in SUSPECT_DATES],
    'primary_evaluation_mask': 'primary_eligible',
    'strict_evaluation_mask': 'history_clean_eligible',
    'special_period_row_interval': '[2021-08-28 18:00, 2021-08-29 12:00)',
    'operating_hour': None, 'final_retrain_suspect_policy': 'not yet selected; candidates only',
    'availability': '24h production plan known at origin; previous hour 60-minute power immediately available',
    'scaling': None, 'missing_value_policy': 'stop if required production or power is missing',
    'weather': 'excluded', 'random_seed_for_downstream_models': 42,
}
(OUT / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
# 저장 후 다시 읽어 팀원이 불러올 때도 행 순서와 값이 유지되는지 확인합니다.
for name, mask in split_masks.items():
    saved_x = pd.read_csv(OUT / f'X_{name}.csv', index_col='sample_id')
    saved_y = pd.read_csv(OUT / f'y_{name}.csv', index_col='sample_id')
    assert saved_x.index.equals(saved_y.index)
    np.testing.assert_allclose(saved_x.to_numpy(), X.loc[mask].to_numpy())
    with np.load(OUT / f'arrays_{name}.npz') as saved:
        np.testing.assert_array_equal(saved['sample_id'], saved_x.index.to_numpy())
        np.testing.assert_allclose(saved['y'].reshape(len(saved_y), -1), saved_y.to_numpy())
print('저장 및 재로딩 검증 완료:', OUT)
'''

In [ ]:
checks = []
def add_check(name, ok, detail=""): checks.append(dict(검사=name, 결과="PASS" if ok else "FAIL", 설명=detail))

# 1) 기준 수치 대조
ref = pd.read_csv(ROOT / "reference/expected_metrics.csv"); GRU_DEP = {"gru_tuned@176", "ens_current@176", "ens_new_w05@176"}
for _, r in ref.iterrows():
    g = METRICS[(METRICS.구간 == r["구간"]) & (METRICS.모델 == r["모델"])].iloc[0]; dep = r["모델"] in GRU_DEP
    wide = dep and r["구간"].startswith("7월")                   # 7월 GRU는 원래 튜닝 노트북의 학습 경로로 만들어져 처음부터 학습한 결과와 조금 다름
    pr_ = .02 if wide else (.01 if dep else .002)
    tol = dict(precision=pr_, recall=pr_, F2=pr_, MAE=.3 if wide else (.2 if dep else .02), FP=300 if wide else (200 if dep else 20))
    diffs = {k: abs(float(g[k]) - float(r[k])) for k in tol}
    add_check(f"기준 수치 대조 | {r['구간']} | {r['모델']}", all(diffs[k] <= tol[k] for k in tol),
              f"F2 {float(g['F2']):.3f} (기준 {r['F2']:.3f}), 정밀도 {float(g['precision']):.3f} ({r['precision']:.3f}), 재현율 {float(g['recall']):.3f} ({r['recall']:.3f}), MAE {float(g['MAE']):.2f} ({r['MAE']:.2f})")

# 1-2) 곡선·2단계 경보·확률 기준 수치 대조
ex = pd.read_csv(ROOT / "reference/expected_extra.csv")


def got_value(item, per):
    cm = CURVE_M[CURVE_M.구간 == per]
    if item == "curve_final MAE": return float(cm[cm.모델.str.startswith("curve_final")].MAE.iloc[0])
    if item == "rule3 MAE": return float(cm[cm.모델.str.startswith("rule3")].MAE.iloc[0])
    if item == "gru MAE": return float(cm[cm.모델 == "gru_tuned"].MAE.iloc[0])
    L = LEVELS[LEVELS.구간 == per].iloc[0]; Pm = PROBM[PROBM.구간 == per].iloc[0]
    return float({"경보 정밀도": L.경보_정밀도, "경보 재현율": L.경보_재현율, "경보와주의 재현율": L.경보와주의_재현율, "놓친 피크(경보와주의)": L.놓친피크_경보와주의, "Brier": Pm.Brier, "ECE": Pm.ECE}[item])


for _, r in ex.iterrows():
    gv = got_value(r["항목"], r["구간"]); add_check(f"기준 수치 대조 | {r['구간']} | {r['항목']}", abs(gv - r["값"]) <= r["허용오차"], f"{gv:.4f} (기준 {r['값']:.4f}, 허용 ±{r['허용오차']})")

# 2) 구조 점검
add_check("창 개수", W.N == 5692 and len(W.ev_val) == 627 and len(W.ev_test) == 1057, f"전체 {W.N}, 평가 val {len(W.ev_val)} / test {len(W.ev_test)}")
add_check("학습 창 개수(val/test 단계)", int(W.tr_val.sum()) == 3985 and int(W.tr_test.sum()) == 4635, f"{int(W.tr_val.sum())} / {int(W.tr_test.sum())}")
add_check("val 학습 창이 7/1 이전에 끝남", W.meta.target_end[W.tr_val].max() <= pd.Timestamp("2021-07-01"), str(W.meta.target_end[W.tr_val].max()))
add_check("test 학습 창이 8/1 이전에 끝남", W.meta.target_end[W.tr_test].max() <= pd.Timestamp("2021-08-01"), str(W.meta.target_end[W.tr_test].max()))
add_check("새 Q1 예측이 평가 창에서 모두 유한값", bool(np.isfinite(Q1_NEW[W.EV]).all()), f"{int(np.isfinite(Q1_NEW[W.EV]).all(axis=1).sum())}/{len(W.EV)} 창")
add_check("경보 기준 이력 창이 모두 해당 주 시작 전에 끝남", len(HIST_LOG) > 0 and all(t_end <= S for S, t_end in HIST_LOG), f"{len(HIST_LOG)}개 평가 주의 이력 창 검사 (8~9월 결과는 기준 선택에 쓰이지 않음)")

# 3a) 미래 생산량 교란: 기준시각+24시간 이후 생산량을 바꿔도 상태 전환 피처가 같은가
rng = np.random.default_rng(0); pick = rng.choice(W.N, 60, replace=False); bad = 0
for w_ in pick:
    sl = slice(w_ * 96, (w_ + 1) * 96)
    base = state_features(W.hi, W.meta["origin"].iloc[[w_]], W.seq[[w_]], W.F.iloc[sl])
    P2 = W.hi.P.copy(); r0 = W.hi.row_of[pd.Timestamp(W.meta["origin"].iloc[w_])]; P2[r0 + 24:] = rng.uniform(0, 3000, len(P2) - r0 - 24)
    pert = state_features(W.hi, W.meta["origin"].iloc[[w_]], W.seq[[w_]], W.F.iloc[sl], P=P2)
    if not np.allclose(base.to_numpy(), pert.to_numpy(), equal_nan=True): bad += 1
add_check("미래 생산량 교란 검사(상태 전환 피처)", bad == 0, f"무작위 {len(pick)}개 창 중 피처가 바뀐 창 {bad}개")

# 3b) 미래 전력 교란: 8/15 이후 전력 원자료를 바꾸고 전처리부터 다시 실행
CUT = pd.Timestamp("2021-08-15")


def run_preprocessing_in(workdir):
    old = os.getcwd(); os.chdir(workdir)
    try:
        g = {"__name__": "__leakcheck__", "display": lambda *a, **k: None}
        with contextlib.redirect_stdout(io.StringIO()): exec(compile(PREPROCESS_SRC, "01_preprocessing(검사용 사본)", "exec"), g)
    finally: os.chdir(old)


tmp = Path(tempfile.mkdtemp(prefix="leakcheck_")); (tmp / "data").mkdir()
rawp = pd.read_csv(ROOT / "data/okm_augumented_2021.csv"); pcols = [c for c in rawp.columns if c.endswith("분")]; mask = rawp["날짜"] >= 20210815
rawp[pcols] = rawp[pcols].astype(float); rr = np.random.default_rng(1); rawp.loc[mask, pcols] = rr.uniform(20, 230, (int(mask.sum()), len(pcols))).round(0)
if "평균" in rawp.columns: rawp["평균"] = rawp["평균"].astype(float); rawp.loc[mask, "평균"] = rawp.loc[mask, pcols].mean(axis=1)
rawp.to_csv(tmp / "data/okm_augumented_2021.csv", index=False); run_preprocessing_in(tmp)
W2 = build_windows(tmp / "results/preprocessing_24h", strict=False)
o1 = {o: i for i, o in enumerate(W.meta.origin)}; o2 = {o: i for i, o in enumerate(W2.meta.origin)}
early = [o for o in W.meta.origin if o < CUT and o in o2]; late = [o for o in W.meta.origin if o >= CUT + pd.Timedelta(days=1) and o in o2]
same = lambda a, b, ids1, ids2: all(np.allclose(a[i1], b[i2], equal_nan=True) for i1, i2 in zip(ids1, ids2))
e1 = [o1[o] for o in early]; e2 = [o2[o] for o in early]
f_early = np.allclose(W.F.to_numpy().reshape(W.N, 96, -1)[e1], W2.F.to_numpy().reshape(W2.N, 96, -1)[e2], equal_nan=True)
f_seq = np.allclose(W.seq[e1], W2.seq[e2], equal_nan=True) and np.allclose(W.rec[e1], W2.rec[e2], equal_nan=True)
l1 = [o1[o] for o in late]; l2 = [o2[o] for o in late]
f_late_changed = not np.allclose(W.rec[l1], W2.rec[l2], equal_nan=True)
add_check("미래 전력 교란 검사(전처리부터 재실행): 입력 배열", f_seq, f"8/15 이전 기준시각 창 {len(early)}개의 sequence·recent 동일")
add_check("미래 전력 교란 검사: 모델 피처 전체(공식 30 + 3단 규칙 + 상태 전환)", f_early, f"8/15 이전 기준시각 창 {len(early)}개 × 96칸 × {W.F.shape[1]}피처 동일")
add_check("교란 검사가 실제로 작동함(8/15 이후 창은 달라짐)", f_late_changed, f"8/16 이후 창 {len(late)}개의 직전 전력 입력이 바뀜")
shutil.rmtree(tmp, ignore_errors=True)

CHK = pd.DataFrame(checks); CHK.to_csv(OUTDIR / "verification_checks.csv", index=False, encoding="utf-8-sig")
display(CHK.assign(설명=CHK.설명.str.slice(0, 120)))
npass = int((CHK.결과 == "PASS").sum()); log(f"자체 검증 {npass}/{len(CHK)} PASS | 전체 실행 시간 {(time.time() - T_START) / 60:.1f}분")
if npass != len(CHK): print("※ FAIL 항목이 있습니다. 위 표의 설명을 확인하세요.")

## 11. 결과 요약과 읽는 법
- `outputs/test_predictions_8_9월.csv`가 8~9월 예측 결과 파일이다. 이 팀의 최종 제안 출력은 **`curve_final`(수치 곡선)**, **`경보단계(0정상/1주의/2경보)`**, **`prob_peak`(피크 확률)**이다(`q1_new`는 경보·확률의 기반이 되는 새 Q1 예측값). `gru_tuned`·`ens_*`·`q1_current`는 비교용이다.
- 지표는 `outputs/metrics_summary.csv`. 피크가 전체의 약 10%라서 경보를 한 번도 안 울려도 정확도가 약 90%이므로(`no_alarm_accuracy`), 정밀도·재현율·F1·F2로 판단한다.
- **한계**: 8~9월은 개발 중 이미 본 구간이다(개발 검증). 평가 주는 7월 5개·8~9월 8개로 적다. 새 Q1의 q·경보 기준 선택 기준이 사후에 바뀌었다(위 소개 참조). 주간 재학습은 이례적인 주(휴무·복귀) 직후에 성능이 흔들릴 수 있다는 점을 `notebooks/`의 L6 실험에서 확인했다.
- 실험 과정(진단, 피처 추가, 분위수 q·경보 기준, 주간 재학습)은 `notebooks/` 의 08~12 노트북에 있다.

현재 최종 선정은 `curve_final` 수치예측과 `최종_피크경보`(개선 Q1 고정 176)입니다. 위 주별 경보단계·확률은 원본 비교 실험 출력으로만 보존합니다.